# vR.P.30.1 --- Image Tampering Detection and Localization
## Multi-quality ELA + CBAM attention (50ep, BCE+Dice)

---

| Field | Value |
|-------|-------|
| **Version** | vR.P.30.1 |
| **Track** | Pretrained Localization (Track 2) |
| **Architecture** | UNet with ResNet-34 encoder (ImageNet pretrained) + **CBAM attention in decoder** |
| **Change** | Multi-quality ELA + CBAM attention (50ep, BCE+Dice) |
| **Parent** | vR.P.30 |
| **Dataset** | CASIA v2.0 --- Splicing Detection + Localization (~12,614 images + GT masks) |
| **Platform** | Google Colab (T4/A100 GPU) |
| **Framework** | PyTorch + Segmentation Models PyTorch (SMP) |

---

### Pipeline Overview

```
Raw RGB Image
    |
    v
Multi-Quality ELA Preprocessing
    |-- JPEG recompress at Q=75 -> grayscale ELA -> Channel 0 (strong artifacts)
    |-- JPEG recompress at Q=85 -> grayscale ELA -> Channel 1 (medium artifacts)
    |-- JPEG recompress at Q=95 -> grayscale ELA -> Channel 2 (subtle artifacts)
    |
    v
3-Channel Multi-Q ELA Map (384x384x3, normalized per-channel)
    |
    v
ResNet-34 Encoder (frozen body + BN unfrozen)
    |
    v
UNet Decoder + **CBAM Attention** (Channel + Spatial attention per decoder block)
    |
    v
Sigmoid -> 384x384 Binary Mask (pixel-level tampered probability)
```

### Components Combined
- **Multi-Q ELA (from P.15):** Q=75/85/95 as 3 independent grayscale channels (+4.09pp F1)
- **CBAM Attention (from P.10):** Channel + Spatial attention in decoder blocks (+3.54pp F1 isolated)



## Project Executive Summary

This section provides a high-level overview of the project for reviewers.
Detailed implementation follows in the numbered sections below.

### Problem Statement

Image tampering -- including copy-move forgery and region splicing -- is
increasingly difficult to detect by visual inspection alone. This project
develops a deep learning system that:

1. **Detects** whether an image has been tampered with (image-level binary classification)
2. **Localizes** the exact tampered pixels by producing a binary segmentation mask (pixel-level localization)

### Dataset Overview

| Property | Value |
|---|---|
| **Dataset** | CASIA v2.0 (publicly available via Kaggle) |
| **Authentic images** | ~7,491 (unmanipulated, all-zero masks) |
| **Tampered images** | ~5,123 (with binary ground truth masks) |
| **Total samples** | ~12,614 |
| **Forgery types** | Copy-move (`Tp_D_*`) and Splicing (`Tp_S_*`) |
| **Ground truth** | Binary masks marking tampered pixel regions |
| **Split** | 70% train / 15% validation / 15% test (stratified) |

### Model Architecture Overview

```
Input: 384x384x3 (3ch MQ-ELA)
        |
   ResNet-34 Encoder (ImageNet pretrained, Frozen body + BN unfrozen)
        |
   UNet Decoder (skip connections)
   CBAM Attention (Channel + Spatial)
        |
        |
   Segmentation Head
        |
   Output: 384x384x1 (tamper probability map)
```

**Key design choices:**

| Feature | Choice | Rationale |
|---|---|---|
| **Encoder** | ResNet-34 (ImageNet) | Strong low-level features for detecting manipulation artifacts |
| **Decoder** | UNet (SMP) | Skip connections preserve spatial resolution for precise masks |
| **Input** | 3-channel Multi-Q ELA (Q=75/85/95 grayscale) | ELA highlights JPEG compression inconsistencies in tampered regions |
| **Freeze** | Frozen body + BN unfrozen | Preserves pretrained features while adapting to ELA domain |
| **Attention** | CBAM (reduction=16, kernel=7) | Channel + spatial attention refines decoder features |

### Training Strategy

| Component | Configuration |
|---|---|
| **Preprocessing** | Multi-quality ELA computation, resize to 384x384 |
| **Loss** | BCE + Dice |
| **Optimizer** | Adam (single LR=1e-3) |
| **Scheduler** | ReduceLROnPlateau (patience=5) |
| **Epochs** | 50 (early stopping patience=10) |
| **AMP** | Enabled (mixed precision for speed + memory) |
| **Key change** | Extended training (50 epochs, patience=10) vs P.30's 25 epochs |

In [ ]:
# ============================================================
# Executive Summary: Final Performance Metrics
# ============================================================
# Displays final test metrics if computed. On first run (top-to-bottom),
# this shows a placeholder. After training + evaluation, re-run to see results.

try:
    print('=' * 60)
    print(f'     FINAL TEST PERFORMANCE -- {VERSION} EXECUTIVE SUMMARY')
    print('=' * 60)
    print()
    print(f'{"Metric":<35} {"Score":>10}')
    print('-' * 47)
    print(f'{"Image-Level Accuracy":<35} {cls_accuracy:.4f}')
    print(f'{"Image-Level ROC-AUC":<35} {cls_auc:.4f}')
    print(f'{"Pixel F1 (Dice)":<35} {pixel_f1:.4f}')
    print(f'{"Pixel IoU":<35} {pixel_iou:.4f}')
    print(f'{"Pixel Precision":<35} {pixel_precision:.4f}')
    print(f'{"Pixel Recall":<35} {pixel_recall:.4f}')
    print(f'{"Pixel AUC":<35} {pixel_auc:.4f}')
    print('-' * 47)
    print()
    print('Note: Tampered-only metrics are the primary evaluation criterion.')
    print('See Evaluation section for full details.')
except NameError:
    print('Final test metrics have not been computed yet.')
    print('Run the full notebook (all sections) to see results here.')
    print()
    print('Expected metrics after training:')
    print('  - Image-Level Accuracy and ROC-AUC')
    print('  - Pixel F1 (Dice) / IoU / Precision / Recall / AUC')

## Results Dashboard

This dashboard provides a condensed overview of the trained model's
performance. It is designed to let reviewers understand the project
quality within a few seconds.

**Note:** On a fresh top-to-bottom run, these cells display placeholder
messages until the training and evaluation sections have completed.
After that, re-running just this section will show live results.

In [ ]:
# ============================================================
# Results Dashboard: Metrics + Training Curves + Sample Prediction
# ============================================================
import matplotlib.pyplot as plt

try:
    fig = plt.figure(figsize=(18, 5))

    # --- Panel 1: Metrics Summary ---
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.axis('off')
    metrics_text = (
        f'FINAL TEST METRICS\n'
        f'-----------------------------------\n'
        f'Pixel F1 (Dice):  {pixel_f1:.4f}\n'
        f'Pixel IoU:        {pixel_iou:.4f}\n'
        f'Pixel Precision:  {pixel_precision:.4f}\n'
        f'Pixel Recall:     {pixel_recall:.4f}\n'
        f'Pixel AUC:        {pixel_auc:.4f}\n'
        f'-----------------------------------\n'
        f'Image Accuracy:   {cls_accuracy:.4f}\n'
        f'Image ROC-AUC:    {cls_auc:.4f}\n'
        f'Image Macro F1:   {cls_macro_f1:.4f}'
    )
    ax1.text(0.1, 0.5, metrics_text, transform=ax1.transAxes,
             fontsize=11, verticalalignment='center', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax1.set_title('Metrics Summary', fontsize=13, fontweight='bold')

    # --- Panel 2: Training Curves ---
    ax2 = fig.add_subplot(1, 3, 2)
    if history.get('train_loss') and len(history['train_loss']) > 0:
        epochs_range = range(1, len(history['train_loss']) + 1)
        ax2.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', alpha=0.7)
        ax2.plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', alpha=0.7)
        ax2.axvline(x=best_epoch, color='green', linestyle='--', alpha=0.5, label=f'Best (ep {best_epoch})')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.legend(fontsize=8)
        ax2.grid(True, alpha=0.3)
    ax2.set_title('Training Curves', fontsize=13, fontweight='bold')

    # --- Panel 3: Example Prediction ---
    ax3 = fig.add_subplot(1, 3, 3)
    _tam_idx = [i for i, l in enumerate(test_labels) if l == 1]
    if _tam_idx:
        _ex_idx = _tam_idx[0]
        _pred_ex = (test_probs[_ex_idx, 0] > 0.5).astype(np.float32)
        _gt_ex = test_masks[_ex_idx, 0]
        # RGB overlay
        _overlay = np.stack([_pred_ex, _gt_ex, np.zeros_like(_pred_ex)], axis=-1)
        ax3.imshow(_overlay)
        ax3.set_title('Example (R=pred, G=GT)', fontsize=11, fontweight='bold')
    ax3.axis('off')

    plt.suptitle(f'{VERSION} -- Results Dashboard', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
except NameError:
    print('Results Dashboard: Run all sections first, then re-run this cell.')

# Table of Contents

0. [Project Executive Summary](#project-executive-summary)
    - [Results Dashboard](#results-dashboard)

1. [Change Log](#change-log)
2. [Setup](#1-setup)
3. [Dataset](#2-dataset) -- CASIA v2.0 discovery, path collection, ELA generation
4. [Data Preparation](#3-data-preparation) -- Transforms, splitting, visualization
5. [Model Architecture](#4-model-architecture) -- UNet + ResNet-34 + CBAM
6. [Training](#5-training) -- BCE + Dice, 50 epochs
7. [Evaluation](#6-evaluation) -- Pixel-level + image-level metrics
8. [Visualization](#7-visualization) -- Predictions, ELA, difference maps
9. [Bonus: Robustness & Tampering Type Analysis](#bonus-robustness--tampering-type-analysis)
10. [Results Summary](#8-results-summary)
11. [Discussion](#9-discussion)
12. [Reproducibility Verification](#reproducibility-verification)
13. [Conclusion](#conclusion)
14. [Save Model](#10-save-model)

## Assignment Requirement Compliance

| # | Requirement | Status | Evidence |
|---|---|---|---|
| 1.1 | Dataset: authentic + tampered + masks | Fulfilled | CASIA v2.0 with IMAGE/MASK directories |
| 1.2 | Data pipeline: cleaning, preprocessing, mask alignment | Fulfilled | Section 2-3: ELA computation, path validation, mask matching |
| 1.3 | Train/val/test split | Fulfilled | 70/15/15 stratified split with leakage verification |
| 1.4 | Data augmentation | Partial | ELA-based preprocessing (augmentation tested in P.30.4) |
| 2.1 | Model architecture for tampered region prediction | Fulfilled | UNet + ResNet-34 encoder + CBAM attention |
| 2.2 | Colab T4 GPU compatible | Fulfilled | Designed for single T4 (15GB VRAM) |
| 3.1 | Performance metrics (localization + detection) | Fulfilled | Pixel F1/IoU/AUC + Image accuracy/AUC/F1 |
| 3.2 | Visual results (Original/GT/Predicted/Overlay) | Fulfilled | Section 7 with multiple visualization types |
| 4.1 | Single Colab notebook | Fulfilled | This notebook |
| 4.2 | Dataset explanation | Fulfilled | Section 2 + Executive Summary |
| 4.3 | Architecture description | Fulfilled | Section 4 + Executive Summary + Model Complexity |
| 4.4 | Training strategy + hyperparameters | Fulfilled | Section 5 with rationale table |
| 4.5 | Evaluation results | Fulfilled | Section 6 + threshold optimization + extended metrics |
| 4.6 | Clear visualizations | Fulfilled | Sections 7 (predictions, ELA, diff maps, failure cases) |
| 4.7 | Model weights | Fulfilled | Saved to Google Drive + HuggingFace Hub |
| B.1 | Robustness testing (JPEG/resize/noise) | Fulfilled | Bonus section: 8 distortion conditions |
| B.2 | Copy-move vs splicing detection | Fulfilled | Bonus section: per-tampering-type breakdown |

---

## Change Log

| Version | Track | Change | Result |
|---------|-------|--------|--------|
| vR.P.3 | Pretrained | ELA as input Q=90 (replace RGB), frozen+BN | Pixel F1 = 0.6920 |
| vR.P.10 | Pretrained | CBAM attention + Focal+Dice | Pixel F1 = 0.7277 |
| vR.P.15 | Pretrained | Multi-quality ELA (Q=75/85/95 as 3 channels) | Pixel F1 = 0.7329 |
| **vR.P.30.1** | **Pretrained** | **Multi-quality ELA + CBAM attention (50ep, BCE+Dice)** | **This notebook** |

### What Changed from vR.P.15
- **Added CBAM attention** (Channel + Spatial) to all 5 UNet decoder blocks
- CBAM operates on decoder feature maps to focus on tampered regions
- **Extended training** to 50 epochs (patience=10)

### What DID NOT change
- IN_CHANNELS: 3 (Multi-Q ELA: Q=75/85/95 as 3 grayscale channels)
- Encoder: ResNet-34 (ImageNet pretrained)
- Data split: 70/15/15 stratified
- Seed: 42



In [ ]:
# ============================================================
# 1. SETUP
# ============================================================

# Install segmentation-models-pytorch
!pip install -q segmentation-models-pytorch
!pip install -q wandb

import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageChops, ImageEnhance
from io import BytesIO
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
import segmentation_models_pytorch as smp

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# ============================================================
# Configuration
# ============================================================
VERSION = 'vR.P.30.1'
CHANGE = 'Multi-quality ELA + CBAM attention (50ep, BCE+Dice)'
SEED = 42
IMAGE_SIZE = 384
BATCH_SIZE = 16
ENCODER = 'resnet34'
ENCODER_WEIGHTS = 'imagenet'
IN_CHANNELS = 3  # ELA is 3-channel RGB
NUM_CLASSES = 1
LEARNING_RATE = 1e-3  # Single LR (decoder + BN only)
ELA_QUALITIES = [75, 85, 95]  # Multi-quality ELA: one grayscale channel per quality
ATTENTION_TYPE = 'cbam'         # CBAM attention in decoder (from P.10)
ATTENTION_REDUCTION = 16        # Channel reduction ratio
CBAM_KERNEL_SIZE = 7            # Spatial attention conv kernel size
DECODER_CHANNELS = (256, 128, 64, 32, 16)  # SMP UNet default for resnet34
EPOCHS = 50
PATIENCE = 10
NUM_WORKERS = 2  # Parallel data loading
CHECKPOINT_PATH = f'{VERSION}_checkpoint.pth'
# --- Kaggle Persistence: Files only ---
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
RESULTS_DIR = '/kaggle/working/results'
LOGS_DIR = '/kaggle/working/logs'
for _d in [CHECKPOINT_DIR, RESULTS_DIR, LOGS_DIR]:
    os.makedirs(_d, exist_ok=True)
RESUME = True  # Kaggle persistence: auto-resume from checkpoints
LATEST_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'latest_checkpoint.pt')
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility

# --- Weights & Biases Experiment Tracking ---
USE_WANDB = True
WANDB_PROJECT = 'Tampered Image Detection & Localization'
DATASET_NAME = 'CASIA2'

import re as _re
_nb_dir = os.path.basename(os.getcwd()).lower()
_run_match = _re.search(r'run-(\d+)', _nb_dir)
RUN_ID = f'run{_run_match.group(1)}' if _run_match else 'run01'
EXPERIMENT_ID = VERSION.lower().replace('.', '').replace(' ', '')

# Infer feature flags from notebook config
_change_lower = CHANGE.lower()
_input_lower = globals().get('INPUT_TYPE', '').lower()
FEATURE_SET = 'rgb'
if 'multi-q' in _input_lower or 'multi-quality' in _change_lower:
    FEATURE_SET = 'multi_quality_ela'
elif 'ela' in _change_lower or 'ela' in _input_lower:
    FEATURE_SET = 'ela'
if 'dct' in _input_lower:
    FEATURE_SET = 'dct' if FEATURE_SET == 'rgb' else f'{FEATURE_SET}+dct'
if 'srm' in _input_lower:
    FEATURE_SET = 'srm_noise'
if 'ycbcr' in _input_lower:
    FEATURE_SET = 'ycbcr'
if 'noiseprint' in _input_lower:
    FEATURE_SET = 'noiseprint'
USE_TTA = 'tta' in _change_lower
JPEG_AUG = 'jpeg' in _change_lower and 'augment' in _change_lower
EDGE_SUPERVISION = 'edge' in _change_lower
NOISE_FEATURES = 'noise' in _input_lower or 'srm' in _input_lower

if USE_WANDB:
    try:
        import wandb
        # Kaggle Secrets first, then interactive fallback
        try:
            from kaggle_secrets import UserSecretsClient
            _key = UserSecretsClient().get_secret("WANDB_API_KEY")
            wandb.login(key=_key)
        except Exception:
            wandb.login()
        wandb.init(
            project=WANDB_PROJECT,
            name=VERSION,
            config={
                'experiment': EXPERIMENT_ID, 'version': VERSION, 'change': CHANGE,
                'run': RUN_ID, 'dataset': DATASET_NAME, 'feature_set': FEATURE_SET,
                'input_type': globals().get('INPUT_TYPE', FEATURE_SET),
                'tta': USE_TTA, 'jpeg_aug': JPEG_AUG,
                'edge_supervision': EDGE_SUPERVISION, 'noise_features': NOISE_FEATURES,
                'encoder': ENCODER, 'in_channels': IN_CHANNELS,
                'img_size': IMAGE_SIZE, 'batch_size': globals().get('BATCH_SIZE', 'N/A'),
                'epochs': globals().get('EPOCHS', 'N/A'), 'learning_rate': globals().get('LEARNING_RATE', 'N/A'), 'patience': globals().get('PATIENCE', 'N/A'),
                'attention_type': ATTENTION_TYPE, 'attention_reduction': ATTENTION_REDUCTION,
                'cbam_kernel_size': CBAM_KERNEL_SIZE,
            },
            reinit=True,
        )
        print(f'W&B run initialized: {EXPERIMENT_ID}_{RUN_ID}')
    except Exception as e:
        print(f'W&B init failed ({e}), continuing without tracking')
        USE_WANDB = False

print(f'Experiment: {EXPERIMENT_ID} | Run: {RUN_ID} | Features: {FEATURE_SET}')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    # TF32 for faster math on Ampere+ GPUs
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# GPU info
print(f"{'='*60}")
print(f"  {VERSION} \u2014 {CHANGE}")
print(f"{'='*60}")
print(f"Device:   {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch:  {torch.__version__}")
print(f"SMP:      {smp.__version__}")
print(f"Image:    {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Input:    Multi-Q ELA (Q={ELA_QUALITIES})")
print(f"Attention: {ATTENTION_TYPE.upper()} (reduction={ATTENTION_REDUCTION}, kernel={CBAM_KERNEL_SIZE})")
print(f"Encoder:  {ENCODER} ({ENCODER_WEIGHTS}, frozen body + BN unfrozen)")
print(f"Batch:    {BATCH_SIZE}")
print(f"LR:       {LEARNING_RATE} (decoder + BN)")
print(f"Epochs:   {EPOCHS}")
print(f"Patience: {PATIENCE}")
print(f"Seed:     {SEED}")
print(f"Workers:  {NUM_WORKERS}")
print(f"AMP:      Enabled (mixed precision)")
print(f"TF32:     Enabled")


---

## 2. Dataset

### CASIA v2.0

The CASIA v2.0 dataset contains ~12,614 images:
- **Authentic (Au):** ~7,491 images â€” unmodified photographs
- **Tampered (Tp):** ~5,123 images â€” images with splicing or copy-move forgery

For **localization**, we need pixel-level ground truth masks indicating which regions are tampered. CASIA v2.0 includes ground truth masks for tampered images. If GT masks are not available in the Kaggle dataset, we generate pseudo-masks using ELA thresholding as a fallback.

For **authentic images**, the ground truth mask is all zeros (no tampering).

In [ ]:
# ============================================================
# 2.1 Dataset Path Discovery (Colab Version)
# ============================================================
# Priority: Google Drive (pre-uploaded) > Kaggle API download
# Expected Google Drive layout:
#   /content/drive/MyDrive/BigVision/datasets/CASIA2/
#       IMAGE/  (contains Au/ and Tp/)
#       MASK/   (contains Au/ and Tp/)

import os
from pathlib import Path

from google.colab import drive

# --- Mount Google Drive ---
GDRIVE_MOUNT = '/content/drive'
if not os.path.isdir(os.path.join(GDRIVE_MOUNT, 'MyDrive')):
    drive.mount(GDRIVE_MOUNT)

GDRIVE_DATASET_DIR = '/content/drive/MyDrive/BigVision/datasets/CASIA2'

def find_dataset():
    """Search for Au/ and Tp/ directories.

    Search order:
      1. Google Drive (GDRIVE_DATASET_DIR)
      2. /content/datasets (Kaggle API download location)
      3. /kaggle/input (if somehow running on Kaggle)

    Returns: (dataset_root, au_dir, tp_dir, gt_mask_dir_or_None)
    """
    search_roots = [
        GDRIVE_DATASET_DIR,
        '/content/datasets',
        '/kaggle/input',
    ]
    candidates = []

    for base in search_roots:
        if not os.path.isdir(base):
            continue
        for dirpath, dirnames, _ in os.walk(base):
            if 'Au' in dirnames and 'Tp' in dirnames:
                candidates.append((
                    dirpath,
                    os.path.join(dirpath, 'Au'),
                    os.path.join(dirpath, 'Tp')
                ))

    if not candidates:
        return None, None, None, None

    # Separate IMAGE vs MASK candidates
    image_candidates = [c for c in candidates if 'mask' not in c[0].lower()]
    mask_candidates  = [c for c in candidates if 'mask' in c[0].lower()]

    if image_candidates:
        explicit_image = [c for c in image_candidates if 'image' in c[0].lower()]
        chosen = explicit_image[0] if explicit_image else image_candidates[0]
    else:
        chosen = candidates[0]

    gt_dir = None
    if mask_candidates:
        gt_dir = mask_candidates[0][0]

    return chosen[0], chosen[1], chosen[2], gt_dir


# --- Try finding the dataset ---
DATASET_ROOT, AU_DIR, TP_DIR, GT_DIR_ROOT = find_dataset()

# --- Copy dataset to local /content/ for fast I/O during training ---
# Google Drive FUSE mount is too slow for per-image reads during training.
LOCAL_DATASET_DIR = '/content/datasets_local'
if DATASET_ROOT is not None and DATASET_ROOT.startswith('/content/drive'):
    if not os.path.isdir(LOCAL_DATASET_DIR):
        import shutil
        print(f'Copying dataset from Google Drive to local storage for fast I/O...')
        shutil.copytree(os.path.dirname(DATASET_ROOT), LOCAL_DATASET_DIR)
        print(f'  Done. Local copy at: {LOCAL_DATASET_DIR}')
    # Re-discover from local copy
    _old_gdrive = GDRIVE_DATASET_DIR
    GDRIVE_DATASET_DIR = LOCAL_DATASET_DIR
    DATASET_ROOT, AU_DIR, TP_DIR, GT_DIR_ROOT = find_dataset()
    GDRIVE_DATASET_DIR = _old_gdrive
    if DATASET_ROOT is not None:
        print(f'Using local copy: {DATASET_ROOT}')
    else:
        print(f'WARNING: Local copy failed, falling back to Google Drive (slow)')
        DATASET_ROOT, AU_DIR, TP_DIR, GT_DIR_ROOT = find_dataset()

# --- Fallback: Download via Kaggle API if not found ---
if DATASET_ROOT is None:
    print("Dataset not found on Google Drive. Downloading via Kaggle API...")
    KAGGLE_DATASET = 'sagnikkayalcse52/casia-spicing-detection-localization'
    DOWNLOAD_DIR = '/content/datasets'

    # Configure Kaggle credentials from Colab secrets
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_KEY')
    except Exception:
        print("WARNING: Could not load Kaggle credentials from Colab secrets.")
        print("Please set KAGGLE_USERNAME and KAGGLE_KEY in Colab secrets,")
        print("or upload your kaggle.json to ~/.kaggle/kaggle.json")

    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'kaggle'], check=True)
    subprocess.run([
        'kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET,
        '-p', DOWNLOAD_DIR, '--unzip'
    ], check=True)

    # Retry discovery after download
    DATASET_ROOT, AU_DIR, TP_DIR, GT_DIR_ROOT = find_dataset()

if DATASET_ROOT is None:
    raise FileNotFoundError(
        'Could not find Au/ and Tp/ directories.\n'
        'Please either:\n'
        f'  1. Upload CASIA2 dataset to Google Drive at: {GDRIVE_DATASET_DIR}\n'
        '  2. Add KAGGLE_USERNAME and KAGGLE_KEY to Colab secrets'
    )

# --- Resolve GT mask directory ---
GT_DIR = None
if GT_DIR_ROOT is not None:
    gt_tp_dir = os.path.join(GT_DIR_ROOT, 'Tp')
    if os.path.isdir(gt_tp_dir):
        gt_files = [f for f in os.listdir(gt_tp_dir)
                    if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.bmp'}]
        if gt_files:
            GT_DIR = GT_DIR_ROOT
            print(f'GT mask structure detected: {GT_DIR}')
            print(f'  MASK/Au: {len(os.listdir(os.path.join(GT_DIR, "Au")))} files')
            print(f'  MASK/Tp: {len(gt_files)} mask files')

if GT_DIR is None:
    search_base = os.path.dirname(DATASET_ROOT)
    for root, dirs, files in os.walk(search_base):
        for d in dirs:
            if any(pat in d.lower() for pat in ['groundtruth', 'gt', 'mask']):
                candidate = os.path.join(root, d)
                if any(Path(f).suffix.lower() in {'.jpg','.jpeg','.png','.tif','.bmp'}
                       for f in os.listdir(candidate) if os.path.isfile(os.path.join(candidate, f))):
                    GT_DIR = candidate
                    break
        if GT_DIR:
            break

SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

print(f'\nDataset root:  {DATASET_ROOT}')
print(f'Authentic dir: {AU_DIR}  ({len(os.listdir(AU_DIR))} files)')
print(f'Tampered dir:  {TP_DIR}  ({len(os.listdir(TP_DIR))} files)')
if GT_DIR:
    print(f'GT mask dir:   {GT_DIR}')
else:
    print(f'GT mask dir:   NOT FOUND \u2014 will generate pseudo-masks from ELA')


In [ ]:
# ============================================================
# 2.2 Collect Image Paths and Ground Truth Masks (UPDATED)
# ============================================================

def collect_paths(directory):
    """Collect sorted image paths from a directory."""
    paths = []
    for f in sorted(os.listdir(directory)):
        if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS:
            paths.append(os.path.join(directory, f))
    return paths

au_paths = collect_paths(AU_DIR)
tp_paths = collect_paths(TP_DIR)

# Build GT mask lookup
# The GT directory may have MASK/Au/ and MASK/Tp/ structure
# OR it may be a flat directory with mask files
gt_map = {}
if GT_DIR:
    gt_tp_dir = os.path.join(GT_DIR, 'Tp')
    gt_au_dir = os.path.join(GT_DIR, 'Au')
    
    # If GT has Au/Tp structure, collect from Tp subdirectory
    if os.path.isdir(gt_tp_dir):
        for f in sorted(os.listdir(gt_tp_dir)):
            if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS:
                stem = Path(f).stem.lower()
                gt_map[stem] = os.path.join(gt_tp_dir, f)
        print(f'GT masks loaded from MASK/Tp/: {len(gt_map)}')
    else:
        # Flat directory
        for f in sorted(os.listdir(GT_DIR)):
            if Path(f).suffix.lower() in SUPPORTED_EXTENSIONS:
                stem = Path(f).stem.lower()
                gt_map[stem] = os.path.join(GT_DIR, f)
        print(f'GT masks loaded (flat): {len(gt_map)}')

# Match tampered images to GT masks
tp_with_gt = 0
tp_without_gt = 0
for tp in tp_paths:
    stem = Path(tp).stem.lower()
    # Try exact match and common variants
    variants = [stem, stem + '_gt', stem.replace('tp', 'gt'), stem.replace('Tp', 'Gt')]
    found = any(v in gt_map for v in variants)
    if found:
        tp_with_gt += 1
    else:
        tp_without_gt += 1

USE_GT_MASKS = GT_DIR is not None and tp_with_gt > len(tp_paths) * 0.5

print(f'\nAuthentic images:  {len(au_paths)}')
print(f'Tampered images:   {len(tp_paths)}')
print(f'Total:             {len(au_paths) + len(tp_paths)}')
print(f'Class ratio (Au:Tp): {len(au_paths)/len(tp_paths):.2f}:1')
if GT_DIR:
    print(f'\nTampered with GT mask:    {tp_with_gt}')
    print(f'Tampered without GT mask: {tp_without_gt}')
print(f'\nUsing GT masks: {USE_GT_MASKS}')
if not USE_GT_MASKS:
    print('  -> Will generate pseudo-masks from ELA thresholding')


In [ ]:
# ============================================================
# 2.3 ELA Pseudo-Mask Generation (Fallback)
# ============================================================

def compute_ela(image_path, quality=90):
    """Compute Error Level Analysis map."""
    original = Image.open(image_path).convert('RGB')
    buffer = BytesIO()
    original.save(buffer, 'JPEG', quality=quality)
    buffer.seek(0)
    resaved = Image.open(buffer)
    ela = ImageChops.difference(original, resaved)
    extrema = ela.getextrema()
    max_diff = max(val[1] for val in extrema)
    if max_diff == 0:
        max_diff = 1
    scale = 255.0 / max_diff
    ela = ImageEnhance.Brightness(ela).enhance(scale)
    return ela

def generate_pseudo_mask(image_path, quality=90, threshold=50):
    """Generate a binary pseudo-mask from ELA.
    Pixels with ELA brightness above threshold are marked as tampered.
    """
    ela = compute_ela(image_path, quality)
    ela_gray = np.array(ela.convert('L'))
    # Adaptive threshold: use mean + 2*std of the ELA map
    mean_val = ela_gray.mean()
    std_val = ela_gray.std()
    adaptive_thresh = max(threshold, mean_val + 2 * std_val)
    mask = (ela_gray > adaptive_thresh).astype(np.float32)
    return mask

def get_gt_mask(image_path, target_size):
    """Get ground truth mask for an image.
    - Authentic images: all-zero mask
    - Tampered images: GT mask if available, else ELA pseudo-mask
    """
    is_tampered = '/Tp/' in image_path or '\\Tp\\' in image_path

    if not is_tampered:
        # Authentic â€” all zeros (no tampering)
        return np.zeros((target_size, target_size), dtype=np.float32)

    if USE_GT_MASKS:
        # Try to find matching GT mask
        stem = Path(image_path).stem.lower()
        variants = [stem, stem + '_gt', stem.replace('tp', 'gt'), stem.replace('Tp', 'Gt')]
        for v in variants:
            if v in gt_map:
                mask = Image.open(gt_map[v]).convert('L')
                mask = mask.resize((target_size, target_size), Image.NEAREST)
                mask_arr = np.array(mask, dtype=np.float32)
                # Normalize to [0, 1]
                if mask_arr.max() > 1:
                    mask_arr = mask_arr / 255.0
                # Binarize
                mask_arr = (mask_arr > 0.5).astype(np.float32)
                return mask_arr

    # Fallback: ELA pseudo-mask
    try:
        mask = generate_pseudo_mask(image_path)
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
        mask_pil = mask_pil.resize((target_size, target_size), Image.NEAREST)
        return np.array(mask_pil, dtype=np.float32) / 255.0
    except Exception:
        return np.zeros((target_size, target_size), dtype=np.float32)

# Quick test
test_au = au_paths[0]
test_tp = tp_paths[0]
mask_au = get_gt_mask(test_au, IMAGE_SIZE)
mask_tp = get_gt_mask(test_tp, IMAGE_SIZE)
print(f'Authentic mask â€” shape: {mask_au.shape}, sum: {mask_au.sum():.0f} (should be 0)')
print(f'Tampered mask  â€” shape: {mask_tp.shape}, sum: {mask_tp.sum():.0f} (should be > 0)')

---

## 3. Data Preparation

### Input Pipeline
- **Multi-quality ELA maps** computed on-the-fly from raw images
  - Channel 0: ELA at Q=75 (grayscale) --- strong compression, highlights major artifacts
  - Channel 1: ELA at Q=85 (grayscale) --- medium compression, balanced sensitivity
  - Channel 2: ELA at Q=95 (grayscale) --- light compression, reveals subtle edits
- Resize each channel to **384x384**
- **Per-channel normalization** (mean/std computed from training set)
- Binary segmentation masks (0=authentic, 1=tampered)
- **70/15/15** stratified split

### Why Multi-Quality ELA?

Different JPEG quality factors expose different artifact magnitudes:

| Quality | Compression | Residual Magnitude | Best For |
|---------|-------------|-------------------|----------|
| Q=75 | Aggressive | Large | Strong manipulations, copy-move |
| Q=85 | Medium | Medium | Balanced detection |
| Q=95 | Light | Small | Subtle splicing, fine boundaries |

Single-quality ELA (Q=90) provides one view. Multi-quality provides three complementary views, like RGB provides three color channels.

### Domain Adaptation
The frozen ResNet-34 encoder was trained on 3-channel RGB images. Multi-quality ELA also has 3 channels, so `conv1` accepts the input without modification. BatchNorm layers adapt to the new per-channel statistics.


In [ ]:
# ============================================================
# 3.1 PyTorch Dataset and Transforms (CHANGED: Multi-Quality ELA)
# ============================================================

def compute_ela_grayscale(image_path, quality):
    """Compute single-quality ELA as a grayscale numpy array [0, 255]."""
    try:
        original = Image.open(image_path).convert('RGB')
        buffer = BytesIO()
        original.save(buffer, 'JPEG', quality=quality)
        buffer.seek(0)
        resaved = Image.open(buffer)
        ela = ImageChops.difference(original, resaved)
        extrema = ela.getextrema()
        max_diff = max(val[1] for val in extrema)
        if max_diff == 0:
            max_diff = 1
        scale = 255.0 / max_diff
        ela = ImageEnhance.Brightness(ela).enhance(scale)
        return np.array(ela.convert('L'))  # grayscale: (H, W)
    except Exception:
        return np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)


def compute_multi_quality_ela(image_path, qualities, size=384):
    """Stack ELA maps at multiple quality levels as a 3-channel image.
    Returns numpy array of shape (size, size, 3) with dtype uint8."""
    channels = []
    for q in qualities:
        gray = compute_ela_grayscale(image_path, q)
        gray_resized = np.array(
            Image.fromarray(gray).resize((size, size), Image.BILINEAR)
        )
        channels.append(gray_resized)
    return np.stack(channels, axis=-1)  # (H, W, 3)


def compute_ela_statistics(image_paths, n_samples=500, size=384, qualities=None):
    """Compute per-channel mean and std of multi-quality ELA maps."""
    if qualities is None:
        qualities = [75, 85, 95]
    rng = np.random.RandomState(42)
    sample_idx = rng.choice(len(image_paths), min(n_samples, len(image_paths)), replace=False)

    channel_sum = np.zeros(3, dtype=np.float64)
    channel_sq_sum = np.zeros(3, dtype=np.float64)
    n_pixels = 0

    for idx in tqdm(sample_idx, desc='Multi-Q ELA stats', leave=False):
        mq_ela = compute_multi_quality_ela(image_paths[idx], qualities, size)
        arr = mq_ela.astype(np.float64) / 255.0  # [0, 1]
        channel_sum += arr.reshape(-1, 3).sum(axis=0)
        channel_sq_sum += (arr.reshape(-1, 3) ** 2).sum(axis=0)
        n_pixels += arr.shape[0] * arr.shape[1]

    mean = channel_sum / n_pixels
    std = np.sqrt(channel_sq_sum / n_pixels - mean ** 2)
    std = np.maximum(std, 1e-5)
    return mean.tolist(), std.tolist()

print('Computing multi-quality ELA normalization statistics...')

# Placeholder --- will be set after split
ELA_MEAN = [0.5, 0.5, 0.5]
ELA_STD = [0.25, 0.25, 0.25]

class CASIASegmentationDataset(Dataset):
    """CASIA v2.0 dataset for tampered region segmentation --- Multi-Quality ELA input."""

    def __init__(self, image_paths, labels, ela_mean, ela_std,
                 mask_size=384, ela_qualities=None):
        self.image_paths = image_paths
        self.labels = labels
        self.mask_size = mask_size
        self.ela_qualities = ela_qualities or [75, 85, 95]
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(mean=ela_mean, std=ela_std)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]

        # Compute multi-quality ELA (3-channel: Q=75, Q=85, Q=95)
        mq_ela = compute_multi_quality_ela(path, self.ela_qualities, self.mask_size)
        # mq_ela is (H, W, 3) uint8

        ela_tensor = self.to_tensor(mq_ela)  # (3, H, W), [0, 1]
        ela_tensor = self.normalize(ela_tensor)  # per-channel normalization

        mask = get_gt_mask(path, self.mask_size)
        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return ela_tensor, mask, label

print(f'Dataset class ready (Multi-Q ELA, Q={ELA_QUALITIES}).')
print(f'ELA stats will be computed after data splitting.')


In [ ]:
# ============================================================
# 3.2 Data Splitting (70/15/15 Stratified) + Multi-Q ELA Stats
# ============================================================

all_paths = au_paths + tp_paths
all_labels = [0] * len(au_paths) + [1] * len(tp_paths)

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED)

# Compute multi-quality ELA normalization statistics from training set
ELA_MEAN, ELA_STD = compute_ela_statistics(
    train_paths, n_samples=500, size=IMAGE_SIZE, qualities=ELA_QUALITIES)
print(f'Multi-Q ELA normalization statistics (from 500 training samples):')
print(f'  Q={ELA_QUALITIES[0]} Mean/Std: {ELA_MEAN[0]:.4f} / {ELA_STD[0]:.4f}')
print(f'  Q={ELA_QUALITIES[1]} Mean/Std: {ELA_MEAN[1]:.4f} / {ELA_STD[1]:.4f}')
print(f'  Q={ELA_QUALITIES[2]} Mean/Std: {ELA_MEAN[2]:.4f} / {ELA_STD[2]:.4f}')

train_dataset = CASIASegmentationDataset(
    train_paths, train_labels, ela_mean=ELA_MEAN, ela_std=ELA_STD,
    mask_size=IMAGE_SIZE, ela_qualities=ELA_QUALITIES)
val_dataset = CASIASegmentationDataset(
    val_paths, val_labels, ela_mean=ELA_MEAN, ela_std=ELA_STD,
    mask_size=IMAGE_SIZE, ela_qualities=ELA_QUALITIES)
test_dataset = CASIASegmentationDataset(
    test_paths, test_labels, ela_mean=ELA_MEAN, ela_std=ELA_STD,
    mask_size=IMAGE_SIZE, ela_qualities=ELA_QUALITIES)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=NUM_WORKERS > 0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         persistent_workers=NUM_WORKERS > 0)

print(f'\nTrain: {len(train_dataset):>6} images  (Au: {sum(1 for l in train_labels if l==0)}, Tp: {sum(1 for l in train_labels if l==1)})')
print(f'Val:   {len(val_dataset):>6} images  (Au: {sum(1 for l in val_labels if l==0)}, Tp: {sum(1 for l in val_labels if l==1)})')
print(f'Test:  {len(test_dataset):>6} images  (Au: {sum(1 for l in test_labels if l==0)}, Tp: {sum(1 for l in test_labels if l==1)})')
print(f'\nTrain batches: {len(train_loader)}  (drop_last=True)')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')
print(f'Workers:       {NUM_WORKERS}  (persistent={NUM_WORKERS > 0})')


In [ ]:
# ============================================================
# Data Leakage Verification
# ============================================================
# Verify zero overlap between train/val/test splits at the path level.

train_paths_set = set(train_dataset.image_paths)
val_paths_set = set(val_dataset.image_paths)
test_paths_set = set(test_dataset.image_paths)

train_val_overlap = train_paths_set & val_paths_set
train_test_overlap = train_paths_set & test_paths_set
val_test_overlap = val_paths_set & test_paths_set

print('=' * 55)
print('  DATA LEAKAGE VERIFICATION')
print('=' * 55)
print()
print(f'Train size: {len(train_paths_set):,}')
print(f'Val size:   {len(val_paths_set):,}')
print(f'Test size:  {len(test_paths_set):,}')
print()
print(f'Train-Val overlap:  {len(train_val_overlap)} images')
print(f'Train-Test overlap: {len(train_test_overlap)} images')
print(f'Val-Test overlap:   {len(val_test_overlap)} images')
print()
if len(train_val_overlap) + len(train_test_overlap) + len(val_test_overlap) == 0:
    print('PASS: No data leakage detected. All splits are disjoint.')
else:
    print('WARNING: Data leakage detected! Check split logic.')

In [ ]:
# ============================================================
# 3.3 Sample Visualization (ELA Input)
# ============================================================

def denormalize_ela(tensor, mean=ELA_MEAN, std=ELA_STD):
    """Reverse ELA normalization for display."""
    t = tensor.clone()
    for ch in range(3):
        t[ch] = t[ch] * std[ch] + mean[ch]
    return t.clamp(0, 1)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

sample_indices = []
au_shown, tp_shown = 0, 0
for i in range(len(train_dataset)):
    if train_labels[i] == 0 and au_shown < 2:
        sample_indices.append(i)
        au_shown += 1
    elif train_labels[i] == 1 and tp_shown < 2:
        sample_indices.append(i)
        tp_shown += 1
    if au_shown >= 2 and tp_shown >= 2:
        break

for row, idx in enumerate(sample_indices):
    ela_tensor, mask, label = train_dataset[idx]
    ela_display = denormalize_ela(ela_tensor).permute(1, 2, 0).numpy()

    # Column 0: ELA input (what the model sees)
    axes[row, 0].imshow(ela_display)
    axes[row, 0].set_title(f'ELA Input ({"Au" if label==0 else "Tp"})', fontsize=10)
    axes[row, 0].axis('off')

    # Column 1: Original RGB (for reference)
    try:
        orig = Image.open(train_paths[idx]).convert('RGB')
        orig = orig.resize((IMAGE_SIZE, IMAGE_SIZE))
        axes[row, 1].imshow(np.array(orig))
    except Exception:
        axes[row, 1].text(0.5, 0.5, 'Load failed', ha='center', va='center')
    axes[row, 1].set_title('Original RGB (reference)', fontsize=10)
    axes[row, 1].axis('off')

    # Column 2: GT Mask
    mask_display = mask.squeeze(0).numpy()
    axes[row, 2].imshow(mask_display, cmap='hot', vmin=0, vmax=1)
    axes[row, 2].set_title(f'GT Mask (sum={mask_display.sum():.0f})', fontsize=10)
    axes[row, 2].axis('off')

    # Column 3: ELA + Mask overlay
    overlay = ela_display.copy()
    mask_rgb = np.zeros_like(overlay)
    mask_rgb[:, :, 0] = mask_display
    overlay = overlay * 0.7 + mask_rgb * 0.3
    overlay = np.clip(overlay, 0, 1)
    axes[row, 3].imshow(overlay)
    axes[row, 3].set_title('ELA + Mask Overlay', fontsize=10)
    axes[row, 3].axis('off')

plt.suptitle('Sample Images: ELA Input | Original RGB | GT Mask | Overlay', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---

## 4. Model Architecture

### UNet with ResNet-34 Encoder (Frozen Body + BN Unfrozen)

```
Input: 384x384x3 (Multi-Q ELA: ch0=Q75, ch1=Q85, ch2=Q95)
|
+-- ENCODER (ResNet-34, ImageNet pretrained)
|   +-- conv1:  7x7, 64, stride 2   -> 192x192x64    [FROZEN, BN UNFROZEN]
|   +-- pool:   3x3, stride 2       -> 96x96x64       [FROZEN]
|   +-- layer1: 3x[3x3, 64]         -> 96x96x64       [FROZEN, BN UNFROZEN]  [skip 1]
|   +-- layer2: 4x[3x3, 128]        -> 48x48x128      [FROZEN, BN UNFROZEN]  [skip 2]
|   +-- layer3: 6x[3x3, 256]        -> 24x24x256      [FROZEN, BN UNFROZEN]  [skip 3]
|   +-- layer4: 3x[3x3, 512]        -> 12x12x512      [FROZEN, BN UNFROZEN]  [skip 4]
|
+-- DECODER (UNet, TRAINABLE, lr=1e-3, ~500K params)
|   +-- up1: 12->24,  concat skip 3  -> 24x24
|   +-- up2: 24->48,  concat skip 2  -> 48x48
|   +-- up3: 48->96,  concat skip 1  -> 96x96
|   +-- up4: 96->192, concat skip 0  -> 192x192
|   +-- final: -> 384x384, 1x1 conv  -> sigmoid
|
Output: 384x384x1 (pixel probability)
```

### Key Design Decisions

| Decision | Choice | Reason |
|----------|--------|--------|
| Input | **Multi-Q ELA (75/85/95)** | 3 independent quality-level signals vs 3 correlated RGB channels |
| Channel 0 (Q=75) | Strong artifacts | Highlights major manipulations |
| Channel 1 (Q=85) | Medium artifacts | Balanced sensitivity |
| Channel 2 (Q=95) | Subtle artifacts | Fine boundary detection |
| Encoder body | **FROZEN** | Conv weights still extract edges/textures |
| Encoder BN | **UNFROZEN** | Adapts to multi-Q ELA statistics |
| IN_CHANNELS | 3 | Compatible with pretrained conv1 (no architecture change) |


In [ ]:

# -- Attention Modules (CBAM from P.10) ------------------------------------

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block \u2014 channel attention only."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        scale = self.squeeze(x).view(b, c)
        scale = self.excitation(scale).view(b, c, 1, 1)
        return x * scale


class ChannelAttention(nn.Module):
    """Channel attention sub-module for CBAM."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _, _ = x.shape
        avg = self.fc(self.avg_pool(x).view(b, c))
        mx = self.fc(self.max_pool(x).view(b, c))
        scale = self.sigmoid(avg + mx).view(b, c, 1, 1)
        return x * scale


class SpatialAttention(nn.Module):
    """Spatial attention sub-module for CBAM."""
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = x.mean(dim=1, keepdim=True)
        mx = x.max(dim=1, keepdim=True)[0]
        descriptor = torch.cat([avg, mx], dim=1)
        attention = self.sigmoid(self.conv(descriptor))
        return x * attention


class CBAMBlock(nn.Module):
    """CBAM: Sequential channel attention then spatial attention."""
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attn = ChannelAttention(channels, reduction)
        self.spatial_attn = SpatialAttention(kernel_size)
    def forward(self, x):
        x = self.channel_attn(x)
        x = self.spatial_attn(x)
        return x


# ============================================================
# 4.1 Build Model (Multi-Q ELA + CBAM, Frozen Body + BN Unfrozen)
# ============================================================

model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=IN_CHANNELS,
    classes=NUM_CLASSES,
    activation=None
)

# Step 1: Freeze ALL encoder parameters
for param in model.encoder.parameters():
    param.requires_grad = False

# Step 2: Unfreeze ONLY BatchNorm layers in encoder (domain adaptation)
bn_param_count = 0
for module in model.encoder.modules():
    if isinstance(module, (nn.BatchNorm2d, nn.SyncBatchNorm)):
        for param in module.parameters():
            param.requires_grad = True
            bn_param_count += param.numel()
        module.track_running_stats = True


# Step 3: Inject CBAM attention into decoder blocks
attn_param_count = 0
if ATTENTION_TYPE is not None:
    for idx_block, block in enumerate(model.decoder.blocks):
        ch = DECODER_CHANNELS[idx_block]
        if ATTENTION_TYPE == 'cbam':
            block.attention2 = CBAMBlock(ch, ATTENTION_REDUCTION, CBAM_KERNEL_SIZE)
        elif ATTENTION_TYPE == 'se':
            block.attention2 = SEBlock(ch, ATTENTION_REDUCTION)
        attn_param_count += sum(p.numel() for p in block.attention2.parameters())
    print(f'Attention type: {ATTENTION_TYPE.upper()}, params: {attn_param_count:,}')


model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
encoder_trainable = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
decoder_trainable = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
seghead_trainable = sum(p.numel() for p in model.segmentation_head.parameters() if p.requires_grad)


print(f'Model: UNet + {ENCODER} ({ENCODER_WEIGHTS}) \u2014 FROZEN BODY + BN UNFROZEN + CBAM')
print(f'Total parameters:     {total_params:>12,}')
print(f'Trainable parameters: {trainable_params:>12,}')
print(f'  Encoder BN params:  {encoder_trainable:>12,}  (BatchNorm only, lr={LEARNING_RATE})')
print(f'  Decoder + CBAM:     {decoder_trainable:>12,}  (lr={LEARNING_RATE})')
print(f'  Segmentation head:  {seghead_trainable:>12,}  (lr={LEARNING_RATE})')
print(f'  CBAM attention:     {attn_param_count:>12,}  ({ATTENTION_TYPE})')
print(f'Frozen parameters:    {frozen_params:>12,}  (all conv/fc weights)')
print(f'Trainable ratio:      {trainable_params/total_params*100:.1f}%')
print(f'Data:param ratio:     1 : {trainable_params/len(train_dataset):.0f}')


In [ ]:
# ============================================================
# Model Complexity Analysis
# ============================================================

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print('=' * 55)
print('  MODEL COMPLEXITY ANALYSIS')
print('=' * 55)
print()
print(f'Total parameters:     {total_params:>12,}')
print(f'Trainable parameters: {trainable_params:>12,}')
print(f'Frozen parameters:    {frozen_params:>12,}')
print(f'Trainable ratio:      {trainable_params/total_params*100:.1f}%')
print()

# Breakdown by component
enc_params = sum(p.numel() for p in model.encoder.parameters())
enc_train = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
dec_params = sum(p.numel() for p in model.decoder.parameters())
dec_train = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
seg_params = sum(p.numel() for p in model.segmentation_head.parameters())

print(f'Component Breakdown:')
print(f'  Encoder:           {enc_params:>10,} ({enc_train:,} trainable)')
print(f'  Decoder:           {dec_params:>10,} ({dec_train:,} trainable)')
print(f'  Segmentation Head: {seg_params:>10,} (all trainable)')
print()
print(f'Data:param ratio:    1 : {trainable_params/len(train_dataset):.0f}')
print()

# Try torchinfo for detailed summary
try:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchinfo'])
    from torchinfo import summary
    print('Detailed model summary (torchinfo):')
    summary(model, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), device=str(DEVICE), verbose=1)
except Exception as e:
    print(f'torchinfo not available: {e}')

---

## 5. Training

### Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| **Loss** | BCE + Dice (combined) | BCE for per-pixel classification; Dice for overlap maximization despite class imbalance |
| **Optimizer** | Adam (weight_decay=1e-5) | Adam adapts LR per-parameter; mild L2 regularization prevents overfitting |
| **LR** | 1e-3 | Same as P.30 baseline; only change is training duration |
| **Scheduler** | ReduceLROnPlateau (factor=0.5, patience=3) | Halves LR when val loss plateaus |
| **Early stopping** | patience=10, monitor=val_loss | Higher patience (10 vs 7) to allow full convergence over 50 epochs |
| **Epochs** | 50 max | Extended from P.30's 25 epochs --- P.30 was still improving at epoch 23 |
| **Batch size** | 16 | Largest stable batch for 384x384x3 on T4 GPU (16 GB VRAM) |
| **Image size** | 384x384 | Balances spatial detail for localization vs. GPU memory |
| **Attention** | CBAM (reduction=16, kernel=7) | Standard CBAM hyperparameters from Woo et al. (2018) |

### Data Augmentation

**No augmentation is applied.** This is deliberate to isolate the effect of extended training:
- P.30 (25ep) vs P.30.1 (50ep) is a controlled comparison --- only training duration changes
- Augmentation is tested separately in vR.P.30.4
- ELA forensic signals are sensitive to intensity augmentations (brightness, blur, noise)

### Why Single LR?

Encoder body is frozen; only BN params, CBAM attention modules, and decoder are trainable.
Extended training (50 epochs) with higher patience (10) allows the model to fully converge.


In [ ]:
# ============================================================
# 5.1 Loss, Optimizer, Scheduler
# ============================================================

bce_loss_fn = smp.losses.SoftBCEWithLogitsLoss()
dice_loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)

def criterion(pred, target):
    return bce_loss_fn(pred, target) + dice_loss_fn(pred, target)

# Single optimizer — all trainable params at same LR
trainable_params_list = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(trainable_params_list, lr=LEARNING_RATE, weight_decay=1e-5)

print(f'Optimizer: Adam (single LR)')
print(f'  Trainable params: {sum(p.numel() for p in trainable_params_list):,}')
print(f'  LR: {LEARNING_RATE}')

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# ============================================================
# 5.2 Training and Validation Functions (AMP-enabled)
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss = 0
    num_batches = 0
    for images, masks, labels in tqdm(loader, desc='Train', leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            predictions = model(images)
            loss = criterion(predictions, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        num_batches += 1
    return total_loss / num_batches

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    num_batches = 0
    all_preds = []
    all_masks = []
    for images, masks, labels in tqdm(loader, desc='Val', leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        with autocast('cuda'):
            predictions = model(images)
            loss = criterion(predictions, masks)
        total_loss += loss.item()
        num_batches += 1
        probs = torch.sigmoid(predictions.float())
        all_preds.append(probs.cpu().numpy())
        all_masks.append(masks.cpu().numpy())
    avg_loss = total_loss / num_batches
    all_preds = np.concatenate(all_preds, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)
    binary_preds = (all_preds > 0.5).astype(np.float32)
    pred_flat = binary_preds.flatten()
    mask_flat = all_masks.flatten()
    eps = 1e-7
    tp = (pred_flat * mask_flat).sum()
    fp = (pred_flat * (1 - mask_flat)).sum()
    fn = ((1 - pred_flat) * mask_flat).sum()
    pixel_f1 = (2 * tp) / (2 * tp + fp + fn + eps)
    pixel_iou = tp / (tp + fp + fn + eps)
    return avg_loss, pixel_f1, pixel_iou

print('Training functions ready (AMP-enabled).')


In [ ]:
# ============================================================
# 5.3 Training Loop (with checkpoint save/resume + AMP)
# ============================================================

scaler = GradScaler('cuda')

history = {
    'train_loss': [], 'val_loss': [], 'val_pixel_f1': [], 'val_pixel_iou': [],
    'lr': []
}

best_val_loss = float('inf')
best_epoch = 0
patience_counter = 0
best_model_state = None
start_epoch = 1

if RESUME and os.path.exists(LATEST_CHECKPOINT):
    print(f'Checkpoint found: {LATEST_CHECKPOINT}')
    ckpt = torch.load(LATEST_CHECKPOINT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(DEVICE)
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['best_val_loss']
    best_epoch = ckpt['best_epoch']
    patience_counter = ckpt['patience_counter']
    history = ckpt['history']
    if ckpt.get('best_model_state') is not None:
        best_model_state = ckpt['best_model_state']
    print(f'  Resuming from epoch {start_epoch} (best_epoch={best_epoch}, best_val_loss={best_val_loss:.4f})')
else:
    print('No checkpoint found \u2014 starting fresh.')

print(f'Starting training: epochs {start_epoch}-{EPOCHS}, patience={PATIENCE}')
print(f'LR: {LEARNING_RATE} | Input: Multi-Q ELA (Q={ELA_QUALITIES}) | AMP: Enabled')
print(f'{"="*80}')

for epoch in range(start_epoch, EPOCHS + 1):
    current_lr = optimizer.param_groups[0]['lr']
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
    val_loss, val_f1, val_iou = validate(model, val_loader, criterion, DEVICE)
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_pixel_f1'].append(val_f1)
    history['val_pixel_iou'].append(val_iou)
    history.get('lr', history.get('lr_encoder', [])).append(current_lr)
    if USE_WANDB:
        wandb.log({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                    'val_pixel_f1': val_f1, 'val_pixel_iou': val_iou, 'lr': current_lr})
    improved = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_model_state, BEST_MODEL_PATH)
        improved = ' *'
    else:
        patience_counter += 1
    print(f'Epoch {epoch:>2}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'Pixel F1: {val_f1:.4f} | IoU: {val_iou:.4f} | LR: {current_lr:.2e}{improved}')
    torch.save({
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_val_loss': best_val_loss, 'best_epoch': best_epoch,
        'patience_counter': patience_counter, 'best_model_state': best_model_state,
        'history': history,
    }, LATEST_CHECKPOINT)
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch}')
        break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    model = model.to(DEVICE)
    print(f'\nRestored best model from epoch {best_epoch} (val_loss={best_val_loss:.4f})')
else:
    print('\nNo improvement during training \u2014 using final weights')

print(f'{"="*80}')
print(f'Training complete. Best epoch: {best_epoch}, Best val loss: {best_val_loss:.4f}')

---

## 6. Evaluation

### Metrics Computed

**Pixel-Level (Localization):**
- Pixel F1 Score (= Dice coefficient)
- IoU (Intersection over Union / Jaccard index)
- Pixel AUC (ROC-AUC on per-pixel probabilities)

**Image-Level (Classification):**
- Accuracy (image classified as tampered if mask has any positive pixel above threshold)
- Per-class Precision, Recall, F1
- Macro F1
- ROC-AUC
- Confusion Matrix

In [ ]:
# ============================================================
# 6.1 Test Set Evaluation â€” Pixel-Level Metrics
# ============================================================

@torch.no_grad()
def evaluate_test(model, loader, device):
    """Full evaluation on test set."""
    model.eval()
    all_probs = []
    all_masks = []
    all_labels = []

    for images, masks, labels in tqdm(loader, desc='Test Eval'):
        images = images.to(device)
        predictions = model(images)
        probs = torch.sigmoid(predictions[0] if isinstance(predictions, tuple) else predictions)

        all_probs.append(probs.cpu().numpy())
        all_masks.append(masks.cpu().numpy())
        all_labels.extend(labels.numpy())

    all_probs = np.concatenate(all_probs, axis=0)  # (N, 1, H, W)
    all_masks = np.concatenate(all_masks, axis=0)   # (N, 1, H, W)
    all_labels = np.array(all_labels)

    return all_probs, all_masks, all_labels

test_probs, test_masks, test_labels = evaluate_test(model, test_loader, DEVICE)
test_preds_binary = (test_probs > 0.5).astype(np.float32)

# Pixel-level metrics (flatten all pixels)
pred_flat = test_preds_binary.flatten()
mask_flat = test_masks.flatten()
prob_flat = test_probs.flatten()

eps = 1e-7
tp = (pred_flat * mask_flat).sum()
fp = (pred_flat * (1 - mask_flat)).sum()
fn = ((1 - pred_flat) * mask_flat).sum()
tn = ((1 - pred_flat) * (1 - mask_flat)).sum()

pixel_f1 = (2 * tp) / (2 * tp + fp + fn + eps)
pixel_iou = tp / (tp + fp + fn + eps)
pixel_dice = pixel_f1  # Dice = F1 for binary
pixel_precision = tp / (tp + fp + eps)
pixel_recall = tp / (tp + fn + eps)

# Pixel AUC (subsample for speed if needed)
n_pixels = len(prob_flat)
if n_pixels > 5_000_000:
    sample_idx = np.random.choice(n_pixels, 5_000_000, replace=False)
    pixel_auc = roc_auc_score(mask_flat[sample_idx], prob_flat[sample_idx])
else:
    pixel_auc = roc_auc_score(mask_flat, prob_flat) if mask_flat.sum() > 0 and (1-mask_flat).sum() > 0 else 0.0

print(f'{"="*60}')
print(f'  PIXEL-LEVEL METRICS (Test Set)')
print(f'{"="*60}')
print(f'  Pixel Precision:  {pixel_precision:.4f}')
print(f'  Pixel Recall:     {pixel_recall:.4f}')
print(f'  Pixel F1 (Dice):  {pixel_f1:.4f}')
print(f'  Pixel IoU:        {pixel_iou:.4f}')
print(f'  Pixel AUC:        {pixel_auc:.4f}')
print(f'{"="*60}')

if USE_WANDB:
    wandb.log({'pixel_f1': pixel_f1, 'pixel_iou': pixel_iou,
               'pixel_precision': pixel_precision, 'pixel_recall': pixel_recall,
               'pixel_auc': pixel_auc})


In [ ]:
# ============================================================
# 6.2 Test Set Evaluation â€” Image-Level Classification
# ============================================================

# Derive image-level classification from masks:
# An image is classified as "tampered" if any predicted pixel > threshold
MASK_AREA_THRESHOLD = 100  # minimum number of tampered pixels to classify as tampered

image_pred_labels = []
image_pred_scores = []

for i in range(len(test_probs)):
    prob_map = test_probs[i, 0]  # (H, W)
    binary_map = (prob_map > 0.5).astype(np.float32)
    tampered_pixel_count = binary_map.sum()

    # Classification: tampered if enough pixels predicted as tampered
    pred_label = 1 if tampered_pixel_count >= MASK_AREA_THRESHOLD else 0
    image_pred_labels.append(pred_label)

    # Score: max probability in the mask (for ROC-AUC)
    image_pred_scores.append(prob_map.max())

image_pred_labels = np.array(image_pred_labels)
image_pred_scores = np.array(image_pred_scores)

# Classification metrics
cls_accuracy = accuracy_score(test_labels, image_pred_labels)
cls_report = classification_report(test_labels, image_pred_labels,
                                     target_names=['Authentic', 'Tampered'],
                                     output_dict=True)
cls_macro_f1 = f1_score(test_labels, image_pred_labels, average='macro')
cls_auc = roc_auc_score(test_labels, image_pred_scores) if len(np.unique(test_labels)) > 1 else 0.0

print(f'{"="*60}')
print(f'  IMAGE-LEVEL CLASSIFICATION (Test Set)')
print(f'{"="*60}')
print(f'  Test Accuracy:    {cls_accuracy:.4f}  ({cls_accuracy*100:.2f}%)')
print(f'  Macro F1:         {cls_macro_f1:.4f}')
print(f'  ROC-AUC:          {cls_auc:.4f}')
print(f'')
print(f'  Per-Class Results:')
print(f'  {"":>12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
for cls_name in ['Authentic', 'Tampered']:
    r = cls_report[cls_name]
    print(f'  {cls_name:>12} {r["precision"]:>10.4f} {r["recall"]:>10.4f} {r["f1-score"]:>10.4f} {r["support"]:>10.0f}')
print(f'{"="*60}')

# Classification report (full)
print('\nFull Classification Report:')
print(classification_report(test_labels, image_pred_labels, target_names=['Authentic', 'Tampered']))

if USE_WANDB:
    wandb.log({'image_accuracy': cls_accuracy, 'image_macro_f1': cls_macro_f1,
               'image_roc_auc': cls_auc})


In [ ]:
# ============================================================
# 6.3 Confusion Matrix and ROC Curve
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(test_labels, image_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Authentic', 'Tampered'],
            yticklabels=['Authentic', 'Tampered'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (Acc={cls_accuracy:.2%})')

# Print confusion details
tn, fp, fn, tp_cls = cm.ravel()
total = cm.sum()
print(f'Confusion Matrix:')
print(f'  TN={tn}, FP={fp}, FN={fn}, TP={tp_cls}')
print(f'  FP Rate: {fp/(tn+fp)*100:.1f}%')
print(f'  FN Rate: {fn/(fn+tp_cls)*100:.1f}%')

# ROC Curve
if len(np.unique(test_labels)) > 1:
    fpr, tpr, thresholds = roc_curve(test_labels, image_pred_scores)
    axes[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {cls_auc:.4f}')
    axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'ROC Curve (AUC={cls_auc:.4f})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{VERSION} â€” Classification Performance', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6.4 Training Curves
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss curves
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-', label='Train Loss')
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-', label='Val Loss')
axes[0, 0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training vs Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Pixel F1
axes[0, 1].plot(epochs_range, history['val_pixel_f1'], 'g-', linewidth=2)
axes[0, 1].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Pixel F1')
axes[0, 1].set_title('Validation Pixel F1')
axes[0, 1].grid(True, alpha=0.3)

# Pixel IoU
axes[1, 0].plot(epochs_range, history['val_pixel_iou'], 'm-', linewidth=2)
axes[1, 0].axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Pixel IoU')
axes[1, 0].set_title('Validation Pixel IoU')
axes[1, 0].grid(True, alpha=0.3)

# Learning Rate
axes[1, 1].plot(epochs_range, history.get('lr', history.get('lr_encoder', [])), 'k-', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'{VERSION} â€” Training History', fontsize=14)
plt.tight_layout()
plt.show()

---

## 7. Visualization

### Original â†’ Ground Truth â†’ Predicted â†’ Overlay

The key deliverable: pixel-level tampered region predictions visualized alongside the original image and ground truth mask.

In [ ]:
# ============================================================
# 7.1 Original / Ground Truth / Predicted / Overlay Grid
# ============================================================

def visualize_predictions(model, dataset, indices, device, title='Predictions'):
    """Display Original | GT Mask | Predicted Mask | Overlay for each index."""
    n = len(indices)
    fig, axes = plt.subplots(n, 4, figsize=(20, 5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    model.eval()

    for row, idx in enumerate(indices):
        img_tensor, gt_mask, label = dataset[idx]

        # Predict
        with torch.no_grad():
            pred_logit = model(img_tensor.unsqueeze(0).to(device))
            pred_logit = pred_logit[0] if isinstance(pred_logit, tuple) else pred_logit
            pred_prob = torch.sigmoid(pred_logit).cpu().squeeze().numpy()

        pred_binary = (pred_prob > 0.5).astype(np.float32)

        # Load original RGB image for display
        rgb_path = dataset.image_paths[idx]
        rgb_img = Image.open(rgb_path).convert('RGB').resize((gt_mask.shape[-1], gt_mask.shape[-2]))
        rgb_display = np.array(rgb_img).astype(np.float32) / 255.0
        gt_display = gt_mask.squeeze(0).numpy()

        # Col 0: Original
        axes[row, 0].imshow(rgb_display)
        axes[row, 0].set_title(f'Original ({"Tampered" if label==1 else "Authentic"})', fontsize=11)
        axes[row, 0].axis('off')

        # Col 1: Ground Truth
        axes[row, 1].imshow(gt_display, cmap='hot', vmin=0, vmax=1)
        axes[row, 1].set_title(f'Ground Truth (sum={gt_display.sum():.0f})', fontsize=11)
        axes[row, 1].axis('off')

        # Col 2: Predicted Mask
        axes[row, 2].imshow(pred_binary, cmap='hot', vmin=0, vmax=1)
        axes[row, 2].set_title(f'Predicted (sum={pred_binary.sum():.0f})', fontsize=11)
        axes[row, 2].axis('off')

        # Col 3: Overlay
        overlay = rgb_display.copy()
        # Green for GT, Red for Predicted
        overlay_mask = np.zeros_like(overlay)
        overlay_mask[:, :, 1] = gt_display * 0.4       # Green = GT
        overlay_mask[:, :, 0] = pred_binary * 0.4       # Red = Predicted
        combined = np.clip(overlay * 0.6 + overlay_mask, 0, 1)
        axes[row, 3].imshow(combined)
        axes[row, 3].set_title('Overlay (green=GT, red=pred)', fontsize=11)
        axes[row, 3].axis('off')

    plt.suptitle(f'{VERSION} â€” {title}', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

# Select sample images: 3 tampered + 2 authentic from test set
tampered_indices = [i for i, l in enumerate(test_labels) if l == 1]
authentic_indices = [i for i, l in enumerate(test_labels) if l == 0]

sample_tp = tampered_indices[:14]
sample_au = authentic_indices[:6]

print('--- Tampered Image Predictions ---')
visualize_predictions(model, test_dataset, sample_tp, DEVICE, 'Tampered Images')

print('\n--- Authentic Image Predictions ---')
visualize_predictions(model, test_dataset, sample_au, DEVICE, 'Authentic Images')

if USE_WANDB:
    try:
        wandb.log({'prediction_examples': wandb.Image(plt.gcf())})
    except Exception:
        pass


In [ ]:
# ============================================================
# 7.2 Per-Image Metric Distribution
# ============================================================

# Compute per-image pixel F1
per_image_f1 = []
per_image_iou = []
per_image_labels = []

for i in range(len(test_probs)):
    pred = (test_probs[i].flatten() > 0.5).astype(np.float32)
    mask = test_masks[i].flatten()

    tp_i = (pred * mask).sum()
    fp_i = (pred * (1 - mask)).sum()
    fn_i = ((1 - pred) * mask).sum()

    f1_i = (2 * tp_i) / (2 * tp_i + fp_i + fn_i + 1e-7)
    iou_i = tp_i / (tp_i + fp_i + fn_i + 1e-7)

    per_image_f1.append(f1_i)
    per_image_iou.append(iou_i)
    per_image_labels.append(test_labels[i])

per_image_f1 = np.array(per_image_f1)
per_image_iou = np.array(per_image_iou)
per_image_labels = np.array(per_image_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 distribution by class
for cls, name in [(0, 'Authentic'), (1, 'Tampered')]:
    mask_cls = per_image_labels == cls
    axes[0].hist(per_image_f1[mask_cls], bins=30, alpha=0.6, label=f'{name} (n={mask_cls.sum()})')
axes[0].set_xlabel('Pixel F1 Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Per-Image Pixel F1 Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU distribution by class
for cls, name in [(0, 'Authentic'), (1, 'Tampered')]:
    mask_cls = per_image_labels == cls
    axes[1].hist(per_image_iou[mask_cls], bins=30, alpha=0.6, label=f'{name} (n={mask_cls.sum()})')
axes[1].set_xlabel('Pixel IoU Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Per-Image Pixel IoU Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{VERSION} â€” Per-Image Metric Distributions', fontsize=14)
plt.tight_layout()
plt.show()

# Summary stats
print(f'Per-Image Pixel F1:')
print(f'  Tampered:  mean={per_image_f1[per_image_labels==1].mean():.4f}, '
      f'std={per_image_f1[per_image_labels==1].std():.4f}')
print(f'  Authentic: mean={per_image_f1[per_image_labels==0].mean():.4f}, '
      f'std={per_image_f1[per_image_labels==0].std():.4f}')

### Segmentation Threshold Optimization

Sweep the segmentation threshold from 0.05 to 0.80 on the **validation set**
and select the threshold maximizing pixel F1. Then re-evaluate the test set
with the optimal threshold for fairer comparison.

This matters because the default 0.5 threshold may not be optimal -- models
often benefit from lower thresholds when recall is more important than precision.

In [ ]:
# ============================================================
# Threshold Sweep on Validation Set
# ============================================================

@torch.no_grad()
def _collect_val_predictions(model, loader, device):
    model.eval()
    all_probs, all_masks, all_labels = [], [], []
    for images, masks, labels in tqdm(loader, desc='Val predictions'):
        images = images.to(device)
        preds = model(images)
        probs = torch.sigmoid(preds[0] if isinstance(preds, tuple) else preds)
        all_probs.append(probs.cpu().numpy())
        all_masks.append(masks.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_masks), np.array(all_labels)

val_probs, val_masks, val_labels = _collect_val_predictions(model, val_loader, DEVICE)

# Sweep thresholds
thresholds = np.arange(0.05, 0.85, 0.02)
best_thr, best_f1_thr = 0.5, 0
thr_results = []

for thr in thresholds:
    _pred = (val_probs > thr).astype(np.float32)
    # Only on tampered images
    _tam_mask = val_labels == 1
    if _tam_mask.sum() == 0:
        continue
    _p = _pred[_tam_mask].flatten()
    _m = val_masks[_tam_mask].flatten()
    _tp = (_p * _m).sum()
    _fp = (_p * (1 - _m)).sum()
    _fn = ((1 - _p) * _m).sum()
    _f1 = (2 * _tp) / (2 * _tp + _fp + _fn + 1e-7)
    thr_results.append((thr, _f1))
    if _f1 > best_f1_thr:
        best_f1_thr = _f1
        best_thr = thr

OPTIMAL_THRESHOLD = best_thr
print(f'Optimal threshold: {OPTIMAL_THRESHOLD:.2f} (val tampered F1 = {best_f1_thr:.4f})')
print(f'Default threshold: 0.50')

# Plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot([t[0] for t in thr_results], [t[1] for t in thr_results], 'b-o', markersize=3)
ax.axvline(x=OPTIMAL_THRESHOLD, color='green', linestyle='--', label=f'Optimal={OPTIMAL_THRESHOLD:.2f}')
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Default=0.50')
ax.set_xlabel('Threshold')
ax.set_ylabel('Tampered Pixel F1')
ax.set_title(f'{VERSION} -- Threshold Optimization (Validation Set)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Re-evaluate test set at optimal threshold
test_preds_opt = (test_probs > OPTIMAL_THRESHOLD).astype(np.float32)
_pf = test_preds_opt.flatten()
_mf = test_masks.flatten()
_tp = (_pf * _mf).sum()
_fp = (_pf * (1 - _mf)).sum()
_fn = ((1 - _pf) * _mf).sum()
opt_f1 = (2 * _tp) / (2 * _tp + _fp + _fn + 1e-7)
opt_iou = _tp / (_tp + _fp + _fn + 1e-7)
print(f'\nTest set at optimal threshold ({OPTIMAL_THRESHOLD:.2f}):')
print(f'  Pixel F1:  {opt_f1:.4f} (vs {pixel_f1:.4f} at 0.50)')
print(f'  Pixel IoU: {opt_iou:.4f} (vs {pixel_iou:.4f} at 0.50)')

In [ ]:
# ============================================================
# Extended Localization Metrics (at Optimal Threshold)
# ============================================================

thr = OPTIMAL_THRESHOLD if 'OPTIMAL_THRESHOLD' in dir() else 0.5
_tam_mask_idx = test_labels == 1

# Tampered-only metrics at optimal threshold
_pred_tam = (test_probs[_tam_mask_idx] > thr).astype(np.float32).flatten()
_gt_tam = test_masks[_tam_mask_idx].flatten()

_tp = (_pred_tam * _gt_tam).sum()
_fp = (_pred_tam * (1 - _gt_tam)).sum()
_fn = ((1 - _pred_tam) * _gt_tam).sum()
_tn = ((1 - _pred_tam) * (1 - _gt_tam)).sum()

_precision = _tp / (_tp + _fp + 1e-7)
_recall = _tp / (_tp + _fn + 1e-7)
_f1 = 2 * _precision * _recall / (_precision + _recall + 1e-7)
_iou = _tp / (_tp + _fp + _fn + 1e-7)
_pixel_acc = (_tp + _tn) / (_tp + _tn + _fp + _fn + 1e-7)

print('=' * 60)
print(f'  EXTENDED LOCALIZATION METRICS (Tampered Only, thr={thr:.2f})')
print('=' * 60)
print(f'  Precision:      {_precision:.4f}')
print(f'  Recall:         {_recall:.4f}')
print(f'  F1 (Dice):      {_f1:.4f}')
print(f'  IoU:            {_iou:.4f}')
print(f'  Pixel Accuracy: {_pixel_acc:.4f}')
print(f'  TP pixels:      {_tp:,.0f}')
print(f'  FP pixels:      {_fp:,.0f}')
print(f'  FN pixels:      {_fn:,.0f}')
print('=' * 60)

In [ ]:
# ============================================================
# Mask-Size Stratified Evaluation
# ============================================================
# Bucket tampered images by mask area to analyze performance across scales.

thr = OPTIMAL_THRESHOLD if 'OPTIMAL_THRESHOLD' in dir() else 0.5
_tam_indices = np.where(test_labels == 1)[0]

BUCKETS = {
    'tiny (<2%)': (0.0, 0.02),
    'small (2-5%)': (0.02, 0.05),
    'medium (5-15%)': (0.05, 0.15),
    'large (>15%)': (0.15, 1.01),
}

bucket_results = {}
for bname, (lo, hi) in BUCKETS.items():
    _f1s, _ious, _count = [], [], 0
    for idx in _tam_indices:
        mask_area = test_masks[idx].sum() / test_masks[idx].size
        if lo <= mask_area < hi:
            _count += 1
            _pred = (test_probs[idx].flatten() > thr).astype(np.float32)
            _gt = test_masks[idx].flatten()
            _tp = (_pred * _gt).sum()
            _fp = (_pred * (1 - _gt)).sum()
            _fn = ((1 - _pred) * _gt).sum()
            _f1 = (2 * _tp) / (2 * _tp + _fp + _fn + 1e-7)
            _iou = _tp / (_tp + _fp + _fn + 1e-7)
            _f1s.append(_f1)
            _ious.append(_iou)
    bucket_results[bname] = {
        'n': _count,
        'f1': np.mean(_f1s) if _f1s else 0,
        'iou': np.mean(_ious) if _ious else 0,
    }

print(f'{"="*65}')
print(f'  MASK-SIZE STRATIFIED EVALUATION (thr={thr:.2f})')
print(f'{"="*65}')
print(f'{"Bucket":<18} {"N":>5} {"Pixel F1":>10} {"Pixel IoU":>10}')
print(f'{"-"*48}')
for bname, r in bucket_results.items():
    print(f'{bname:<18} {r["n"]:5d} {r["f1"]:10.4f} {r["iou"]:10.4f}')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
_names = [k for k in bucket_results if bucket_results[k]['n'] > 0]
_f1s = [bucket_results[k]['f1'] for k in _names]
_ious = [bucket_results[k]['iou'] for k in _names]
_colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

for ax, vals, title in [(axes[0], _f1s, 'Pixel F1'), (axes[1], _ious, 'Pixel IoU')]:
    bars = ax.bar(_names, vals, color=_colors[:len(_names)])
    ax.set_title(f'{VERSION} -- {title} by Mask Size')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=15)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}',
                ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Failure Case Analysis (10 Worst Predictions)
# ============================================================

thr = OPTIMAL_THRESHOLD if 'OPTIMAL_THRESHOLD' in dir() else 0.5
_tam_indices = np.where(test_labels == 1)[0]

# Compute per-image Dice for tampered images
_per_dice = []
for idx in _tam_indices:
    _pred = (test_probs[idx].flatten() > thr).astype(np.float32)
    _gt = test_masks[idx].flatten()
    _tp = (_pred * _gt).sum()
    _fp = (_pred * (1 - _gt)).sum()
    _fn = ((1 - _pred) * _gt).sum()
    _dice = (2 * _tp) / (2 * _tp + _fp + _fn + 1e-7)
    _per_dice.append((idx, _dice))

# Sort by worst Dice
_per_dice.sort(key=lambda x: x[1])
_worst_10 = _per_dice[:10]

print(f'10 Worst Predictions (by Dice):')
print(f'{"Rank":<5} {"Index":>6} {"Dice":>8} {"Image":<40}')
print('-' * 65)
for rank, (idx, dice) in enumerate(_worst_10, 1):
    fname = os.path.basename(test_dataset.image_paths[idx])
    print(f'{rank:<5} {idx:6d} {dice:8.4f} {fname}')

# Visualize worst 5
n_show = min(5, len(_worst_10))
fig, axes = plt.subplots(n_show, 4, figsize=(20, 5 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for row, (idx, dice) in enumerate(_worst_10[:n_show]):
    _pred_map = (test_probs[idx, 0] > thr).astype(np.float32)
    _gt_map = test_masks[idx, 0]

    # Load original image for display
    _rgb = Image.open(test_dataset.image_paths[idx]).convert('RGB')
    _rgb = _rgb.resize((_gt_map.shape[1], _gt_map.shape[0]))
    _rgb_arr = np.array(_rgb).astype(np.float32) / 255.0

    axes[row, 0].imshow(_rgb_arr)
    axes[row, 0].set_title(f'Original (#{idx})', fontsize=10)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(_gt_map, cmap='hot', vmin=0, vmax=1)
    axes[row, 1].set_title(f'Ground Truth', fontsize=10)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(_pred_map, cmap='hot', vmin=0, vmax=1)
    axes[row, 2].set_title(f'Predicted (Dice={dice:.3f})', fontsize=10)
    axes[row, 2].axis('off')

    # Overlay: green=GT, red=pred, yellow=overlap
    _overlay = _rgb_arr.copy()
    _overlay_mask = np.zeros_like(_overlay)
    _overlay_mask[:, :, 1] = _gt_map * 0.4     # Green = GT
    _overlay_mask[:, :, 0] = _pred_map * 0.4    # Red = Predicted
    _combined = np.clip(_overlay * 0.6 + _overlay_mask, 0, 1)
    axes[row, 3].imshow(_combined)
    axes[row, 3].set_title('Overlay (G=GT, R=Pred)', fontsize=10)
    axes[row, 3].axis('off')

plt.suptitle(f'{VERSION} -- Failure Cases (5 Worst by Dice)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ELA Channel Visualization (Multi-Quality ELA)
# ============================================================
# Show what the model sees: the 3 ELA quality channels alongside predictions.

model.eval()
_tam_idx = [i for i, l in enumerate(test_labels) if l == 1]
_n_show = min(4, len(_tam_idx))

fig, axes = plt.subplots(_n_show, 5, figsize=(22, 5 * _n_show))
if _n_show == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(_tam_idx[:_n_show]):
    img_tensor, gt_mask, label = test_dataset[idx]

    # Denormalize ELA channels for display
    _ela_raw = img_tensor.clone()
    for c in range(3):
        _ela_raw[c] = _ela_raw[c] * ela_std[c] + ela_mean[c]
    _ela_raw = _ela_raw.clamp(0, 1).numpy()

    # Predict
    with torch.no_grad():
        pred = model(img_tensor.unsqueeze(0).to(DEVICE))
        pred = pred[0] if isinstance(pred, tuple) else pred
        prob = torch.sigmoid(pred).cpu().squeeze().numpy()
    pred_binary = (prob > 0.5).astype(np.float32)
    gt_np = gt_mask.squeeze().numpy()

    # Original RGB
    _rgb = Image.open(test_dataset.image_paths[idx]).convert('RGB')
    _rgb = _rgb.resize((gt_np.shape[1], gt_np.shape[0]))
    _rgb_arr = np.array(_rgb).astype(np.float32) / 255.0

    axes[row, 0].imshow(_rgb_arr)
    axes[row, 0].set_title('Original RGB', fontsize=10)
    axes[row, 0].axis('off')

    # ELA composite (all 3 channels as RGB)
    _ela_display = np.stack([_ela_raw[0], _ela_raw[1], _ela_raw[2]], axis=-1)
    axes[row, 1].imshow(np.clip(_ela_display, 0, 1))
    axes[row, 1].set_title('ELA (Q75/Q85/Q95 as RGB)', fontsize=10)
    axes[row, 1].axis('off')

    # Ground truth
    axes[row, 2].imshow(gt_np, cmap='hot', vmin=0, vmax=1)
    axes[row, 2].set_title('Ground Truth', fontsize=10)
    axes[row, 2].axis('off')

    # Predicted
    axes[row, 3].imshow(pred_binary, cmap='hot', vmin=0, vmax=1)
    axes[row, 3].set_title('Predicted Mask', fontsize=10)
    axes[row, 3].axis('off')

    # ELA difference highlight
    _ela_sum = _ela_raw.mean(axis=0)  # average across Q levels
    axes[row, 4].imshow(_ela_sum, cmap='jet')
    axes[row, 4].set_title('ELA Intensity (avg)', fontsize=10)
    axes[row, 4].axis('off')

plt.suptitle(f'{VERSION} -- ELA Channel Visualization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Difference Map and Contour Overlay Visualization
# ============================================================
import cv2

thr = OPTIMAL_THRESHOLD if 'OPTIMAL_THRESHOLD' in dir() else 0.5
_tam_idx = [i for i, l in enumerate(test_labels) if l == 1]
_n_show = min(4, len(_tam_idx))

fig, axes = plt.subplots(_n_show, 6, figsize=(26, 4.5 * _n_show))
if _n_show == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(_tam_idx[:_n_show]):
    _pred_map = (test_probs[idx, 0] > thr).astype(np.float32)
    _prob_map = test_probs[idx, 0]
    _gt_map = test_masks[idx, 0]

    # Load original
    _rgb = Image.open(test_dataset.image_paths[idx]).convert('RGB')
    _rgb = _rgb.resize((_gt_map.shape[1], _gt_map.shape[0]))
    _rgb_arr = np.array(_rgb).astype(np.float32) / 255.0

    # Col 0: Original
    axes[row, 0].imshow(_rgb_arr)
    axes[row, 0].set_title('Original', fontsize=10)
    axes[row, 0].axis('off')

    # Col 1: Ground Truth
    axes[row, 1].imshow(_gt_map, cmap='hot', vmin=0, vmax=1)
    axes[row, 1].set_title('Ground Truth', fontsize=10)
    axes[row, 1].axis('off')

    # Col 2: Predicted
    axes[row, 2].imshow(_pred_map, cmap='hot', vmin=0, vmax=1)
    axes[row, 2].set_title('Predicted', fontsize=10)
    axes[row, 2].axis('off')

    # Col 3: Overlay (green=GT, red=pred)
    _overlay = _rgb_arr.copy()
    _om = np.zeros_like(_overlay)
    _om[:, :, 1] = _gt_map * 0.4
    _om[:, :, 0] = _pred_map * 0.4
    axes[row, 3].imshow(np.clip(_overlay * 0.6 + _om, 0, 1))
    axes[row, 3].set_title('Overlay', fontsize=10)
    axes[row, 3].axis('off')

    # Col 4: Difference Map
    _diff = np.abs(_pred_map - _gt_map)
    axes[row, 4].imshow(_diff, cmap='RdYlGn_r', vmin=0, vmax=1)
    axes[row, 4].set_title(f'Difference (err={_diff.mean():.3f})', fontsize=10)
    axes[row, 4].axis('off')

    # Col 5: Contour Overlay
    _contour_img = (_rgb_arr * 255).astype(np.uint8).copy()
    _gt_u8 = (_gt_map * 255).astype(np.uint8)
    _pred_u8 = (_pred_map * 255).astype(np.uint8)
    gt_contours, _ = cv2.findContours(_gt_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    pred_contours, _ = cv2.findContours(_pred_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(_contour_img, gt_contours, -1, (0, 255, 0), 2)   # Green = GT
    cv2.drawContours(_contour_img, pred_contours, -1, (255, 0, 0), 2)  # Red = Pred
    axes[row, 5].imshow(_contour_img)
    axes[row, 5].set_title('Contours (G=GT, R=Pred)', fontsize=10)
    axes[row, 5].axis('off')

plt.suptitle(f'{VERSION} -- Difference Maps & Contour Overlays', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Bonus: Robustness & Tampering Type Analysis

### Robustness Evaluation
Tests model degradation under common post-processing distortions:
JPEG compression (Q=50/70/90), spatial resampling (0.5x/0.75x), and Gaussian noise (sigma=5/15).

### Per-Tampering-Type Breakdown
CASIA v2.0 encodes tampering type in filenames:
- `Tp_D_*` = **Copy-move** (duplicate region within image)
- `Tp_S_*` = **Splicing** (paste from external image)

Metrics are split by type to assess model strengths and weaknesses.


In [ ]:
# ============================================================
# Bonus 1: Robustness Against Distortions
# ============================================================
# Tests: JPEG compression, spatial resampling, Gaussian noise

import tempfile

def _jpeg_compress(img, quality):
    """Apply JPEG compression to PIL image."""
    buf = BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

def _resize_back(img, scale):
    """Downscale then upscale (simulates resampling artifacts)."""
    w, h = img.size
    small = img.resize((int(w * scale), int(h * scale)), Image.BILINEAR)
    return small.resize((w, h), Image.BILINEAR)

def _gaussian_noise(img, sigma):
    """Add Gaussian noise to PIL image."""
    arr = np.array(img, dtype=np.float32)
    noise = np.random.normal(0, sigma, arr.shape).astype(np.float32)
    return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

DISTORTIONS = {
    'Clean (baseline)': lambda img: img,
    'JPEG Q=50':        lambda img: _jpeg_compress(img, 50),
    'JPEG Q=70':        lambda img: _jpeg_compress(img, 70),
    'JPEG Q=90':        lambda img: _jpeg_compress(img, 90),
    'Resize 0.5x':      lambda img: _resize_back(img, 0.5),
    'Resize 0.75x':     lambda img: _resize_back(img, 0.75),
    'Noise \u03c3=5':   lambda img: _gaussian_noise(img, 5),
    'Noise \u03c3=15':  lambda img: _gaussian_noise(img, 15),
}

# Sample tampered test images (cap at 200 for speed)
_tampered_idx = [i for i, l in enumerate(test_labels) if l == 1]
_N_ROBUST = min(200, len(_tampered_idx))
_robust_sample = _tampered_idx[:_N_ROBUST]
MASK_AREA_THRESHOLD = 100  # pixels for image-level detection

print(f'Robustness evaluation: {_N_ROBUST} tampered test images x {len(DISTORTIONS)} distortions')
print(f'{"="*70}')

model.eval()
robustness_results = {}

for dist_name, dist_fn in DISTORTIONS.items():
    _pixel_f1s = []
    _correct = 0

    for idx in tqdm(_robust_sample, desc=dist_name, leave=False):
        orig_path = test_dataset.image_paths[idx]

        if dist_name == 'Clean (baseline)':
            # Use original pipeline directly
            img_tensor, gt_mask, label = test_dataset[idx]
        else:
            # Distort image, save to temp, swap path, run pipeline
            _raw = Image.open(orig_path).convert('RGB')
            _distorted = dist_fn(_raw)
            with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as _tmp:
                _distorted.save(_tmp, format='PNG')
                _tmp_path = _tmp.name
            test_dataset.image_paths[idx] = _tmp_path
            try:
                img_tensor, gt_mask, label = test_dataset[idx]
            finally:
                test_dataset.image_paths[idx] = orig_path
                os.unlink(_tmp_path)

        # Predict
        with torch.no_grad():
            pred = model(img_tensor.unsqueeze(0).to(DEVICE))
            pred = pred[0] if isinstance(pred, tuple) else pred
            prob = torch.sigmoid(pred).cpu().squeeze().numpy()

        pred_binary = (prob > 0.5).astype(np.float32)
        gt_np = gt_mask.squeeze().numpy()

        # Pixel F1
        if gt_np.sum() > 0:
            inter = (pred_binary * gt_np).sum()
            prec = inter / (pred_binary.sum() + 1e-8)
            rec = inter / (gt_np.sum() + 1e-8)
            f1 = 2 * prec * rec / (prec + rec + 1e-8)
            _pixel_f1s.append(f1)

        # Image-level detection
        if pred_binary.sum() > MASK_AREA_THRESHOLD:
            _correct += 1

    _mean_f1 = np.mean(_pixel_f1s) if _pixel_f1s else 0
    _recall = _correct / _N_ROBUST
    robustness_results[dist_name] = {'pixel_f1': _mean_f1, 'tampered_recall': _recall}
    print(f'{dist_name:20s}  Pixel F1={_mean_f1:.4f}  Tampered Recall={_recall:.4f}')

# --- Bar chart ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(robustness_results.keys())
f1_vals = [robustness_results[n]['pixel_f1'] for n in names]
recall_vals = [robustness_results[n]['tampered_recall'] for n in names]

colors = ['#2ecc71'] + ['#e74c3c']*3 + ['#3498db']*2 + ['#f39c12']*2  # green=clean, red=jpeg, blue=resize, orange=noise

axes[0].barh(names, f1_vals, color=colors)
axes[0].set_xlabel('Pixel F1')
axes[0].set_title(f'{VERSION} \u2014 Robustness: Pixel F1')
axes[0].set_xlim(0, 1)
for i, v in enumerate(f1_vals):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

axes[1].barh(names, recall_vals, color=colors)
axes[1].set_xlabel('Tampered Recall')
axes[1].set_title(f'{VERSION} \u2014 Robustness: Detection Recall')
axes[1].set_xlim(0, 1)
for i, v in enumerate(recall_vals):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

if USE_WANDB:
    try:
        _tbl = wandb.Table(columns=['Distortion', 'Pixel F1', 'Tampered Recall'])
        for n in names:
            _tbl.add_data(n, robustness_results[n]['pixel_f1'], robustness_results[n]['tampered_recall'])
        wandb.log({'robustness_table': _tbl, 'robustness_chart': wandb.Image(fig)})
    except Exception:
        pass

# --- Summary ---
baseline_f1 = robustness_results['Clean (baseline)']['pixel_f1']
print(f'\n{"="*70}')
print(f'Baseline Pixel F1: {baseline_f1:.4f}')
print(f'Worst degradation:')
worst = min(robustness_results.items(), key=lambda x: x[1]['pixel_f1'] if x[0] != 'Clean (baseline)' else 999)
print(f'  {worst[0]}: F1={worst[1]["pixel_f1"]:.4f} (delta={worst[1]["pixel_f1"] - baseline_f1:+.4f})')


In [ ]:
# ============================================================
# Bonus 2: Per-Tampering-Type Breakdown (Copy-Move vs Splicing)
# ============================================================
# CASIA v2.0 filename convention:
#   Tp_D_* = Copy-move (Duplicate)
#   Tp_S_* = Splicing

def _get_tampering_type(path):
    """Extract tampering type from CASIA filename."""
    fname = os.path.basename(path)
    if fname.startswith('Tp_D') or '_D_' in fname:
        return 'copy-move'
    elif fname.startswith('Tp_S') or '_S_' in fname:
        return 'splicing'
    elif fname.startswith('Au'):
        return 'authentic'
    return 'unknown'

# Classify all test images by type
_type_indices = {'copy-move': [], 'splicing': [], 'authentic': [], 'unknown': []}
for i in range(len(test_dataset)):
    _t = _get_tampering_type(test_dataset.image_paths[i])
    _type_indices[_t].append(i)

print(f'Test set composition:')
for _t, _idxs in _type_indices.items():
    if _idxs:
        print(f'  {_t:12s}: {len(_idxs)} images')

# Evaluate per type
model.eval()
type_results = {}

for _type_name in ['copy-move', 'splicing']:
    _idxs = _type_indices[_type_name]
    if not _idxs:
        print(f'\n{_type_name}: No images found, skipping.')
        continue

    _pf1s, _ious = [], []
    _img_correct = 0

    for idx in tqdm(_idxs, desc=_type_name, leave=False):
        img_tensor, gt_mask, label = test_dataset[idx]

        with torch.no_grad():
            pred = model(img_tensor.unsqueeze(0).to(DEVICE))
            pred = pred[0] if isinstance(pred, tuple) else pred
            prob = torch.sigmoid(pred).cpu().squeeze().numpy()

        pred_binary = (prob > 0.5).astype(np.float32)
        gt_np = gt_mask.squeeze().numpy()

        # Pixel F1 and IoU
        if gt_np.sum() > 0:
            inter = (pred_binary * gt_np).sum()
            union = pred_binary.sum() + gt_np.sum() - inter
            prec = inter / (pred_binary.sum() + 1e-8)
            rec = inter / (gt_np.sum() + 1e-8)
            f1 = 2 * prec * rec / (prec + rec + 1e-8)
            iou = inter / (union + 1e-8)
            _pf1s.append(f1)
            _ious.append(iou)

        # Image-level
        if pred_binary.sum() > MASK_AREA_THRESHOLD:
            _img_correct += 1

    type_results[_type_name] = {
        'n': len(_idxs),
        'pixel_f1': np.mean(_pf1s) if _pf1s else 0,
        'pixel_iou': np.mean(_ious) if _ious else 0,
        'recall': _img_correct / len(_idxs),
    }

# --- Display results ---
print(f'\n{"="*70}')
print(f'{VERSION} \u2014 Per-Tampering-Type Results')
print(f'{"="*70}')
print(f'{"Type":15s} {"N":>6s} {"Pixel F1":>10s} {"Pixel IoU":>10s} {"Recall":>10s}')
print(f'{"-"*55}')
for _t, _r in type_results.items():
    print(f'{_t:15s} {_r["n"]:6d} {_r["pixel_f1"]:10.4f} {_r["pixel_iou"]:10.4f} {_r["recall"]:10.4f}')

# --- Bar chart ---
if len(type_results) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    _types = list(type_results.keys())
    _colors = ['#e74c3c', '#3498db']

    for ax, metric, title in zip(axes,
        ['pixel_f1', 'pixel_iou', 'recall'],
        ['Pixel F1', 'Pixel IoU', 'Detection Recall']):
        vals = [type_results[t][metric] for t in _types]
        bars = ax.bar(_types, vals, color=_colors[:len(_types)])
        ax.set_title(f'{VERSION} \u2014 {title}')
        ax.set_ylim(0, 1)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}',
                    ha='center', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    if USE_WANDB:
        try:
            _tbl = wandb.Table(columns=['Type', 'N', 'Pixel F1', 'Pixel IoU', 'Recall'])
            for _t, _r in type_results.items():
                _tbl.add_data(_t, _r['n'], _r['pixel_f1'], _r['pixel_iou'], _r['recall'])
            wandb.log({'tampering_type_table': _tbl, 'tampering_type_chart': wandb.Image(fig)})
        except Exception:
            pass

print(f'\nKey insight: ', end='')
if len(type_results) >= 2:
    _types = list(type_results.keys())
    _f1s = [type_results[t]['pixel_f1'] for t in _types]
    _better = _types[np.argmax(_f1s)]
    _worse = _types[np.argmin(_f1s)]
    _gap = max(_f1s) - min(_f1s)
    print(f'{_better} ({max(_f1s):.4f}) outperforms {_worse} ({min(_f1s):.4f}) by {_gap:.4f} F1')


In [ ]:
# ============================================================
# Inference Speed Benchmark
# ============================================================
import time

N_BENCH = min(50, len(test_dataset))
model.eval()

# Warmup
with torch.no_grad():
    _dummy = test_dataset[0][0].unsqueeze(0).to(DEVICE)
    for _ in range(3):
        _ = model(_dummy)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# Benchmark
latencies = []
for i in range(N_BENCH):
    img_tensor = test_dataset[i][0].unsqueeze(0).to(DEVICE)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = model(img_tensor)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000)

latencies = np.array(latencies)
print('=' * 55)
print('  INFERENCE SPEED BENCHMARK')
print('=' * 55)
print(f'  Images tested:  {N_BENCH}')
print(f'  Mean latency:   {latencies.mean():.1f} ms')
print(f'  Median latency: {np.median(latencies):.1f} ms')
print(f'  Std:            {latencies.std():.1f} ms')
print(f'  Throughput:     {1000.0 / latencies.mean():.1f} images/sec')
print(f'  Min / Max:      {latencies.min():.1f} / {latencies.max():.1f} ms')
print('=' * 55)

---

## 8. Results Summary

In [ ]:
# ============================================================
# 8.1 Results Tracking Table
# ============================================================

print(f'{"="*100}')
print(f'  RESULTS SUMMARY --- vR.P.30.1')
print(f'{"="*100}')
print()

# Ablation comparison table
print('Cross-Track Comparison:')
print(f'{"Version":<10} {"Track":<12} {"Encoder":<12} {"Input":<14} '
      f'{"Test Acc":<10} {"Pixel-F1":<10} {"IoU":<8}')
print('-' * 84)

print(f'{"vR.P.3":<10} {"Pretrained":<12} {"ResNet-34":<12} {"ELA Q90 384sq":<14} '
      f'{"86.79%":<10} {"0.6920":<10} {"0.5291":<8}')

print(f'{"vR.P.10":<10} {"Pretrained":<12} {"ResNet-34":<12} {"ELA+CBAM":<14} '
      f'{"87.32%":<10} {"0.7277":<10} {"0.5719":<8}')

print(f'{"vR.P.15":<10} {"Pretrained":<12} {"ResNet-34":<12} {"MQ-ELA 384sq":<14} '
      f'{"87.53%":<10} {"0.7329":<10} {"0.5785":<8}')

# This run
print(f'{"vR.P.30.1":<10} {"Pretrained":<12} {"ResNet-34":<12} {"MQ-ELA+CBAM":<14} '
      f'{cls_accuracy*100:.2f}{"%-":<7} {pixel_f1:<10.4f} {pixel_iou:<8.4f}')
print()

# Full metrics
print(f'Pixel-Level Metrics:')
print(f'  Pixel F1 (Dice): {pixel_f1:.4f}')
print(f'  Pixel IoU:       {pixel_iou:.4f}')
print(f'  Pixel Precision: {pixel_precision:.4f}')
print(f'  Pixel Recall:    {pixel_recall:.4f}')
print(f'  Pixel AUC:       {pixel_auc:.4f}')
print()
print(f'Image-Level Metrics:')
print(f'  Accuracy:        {cls_accuracy:.4f} ({cls_accuracy*100:.2f}%)')
print(f'  Macro F1:        {cls_macro_f1:.4f}')
print(f'  ROC-AUC:         {cls_auc:.4f}')
print(f'  Au Precision:    {cls_report["Authentic"]["precision"]:.4f}')
print(f'  Au Recall:       {cls_report["Authentic"]["recall"]:.4f}')
print(f'  Au F1:           {cls_report["Authentic"]["f1-score"]:.4f}')
print(f'  Tp Precision:    {cls_report["Tampered"]["precision"]:.4f}')
print(f'  Tp Recall:       {cls_report["Tampered"]["recall"]:.4f}')
print(f'  Tp F1:           {cls_report["Tampered"]["f1-score"]:.4f}')
print()
print(f'Training:')
print(f'  Best epoch:      {best_epoch}')
print(f'  Epochs trained:  {len(history["train_loss"])}')
print(f'  Best val loss:   {best_val_loss:.4f}')
print(f'{"="*100}')


---

## 9. Discussion

### What Changed in vR.P.30.1

**Extended Training (50ep, patience=10):** Same architecture as P.30 (Multi-Q ELA + CBAM)
but trained for 50 epochs with higher early stopping patience to allow full convergence.

### Key Findings

- **Pixel F1 = 0.7762** (+3.24pp over P.30's 0.7438) --- extended training clearly helps
- **Pixel IoU = 0.6343**, **Pixel AUC = 0.9795**
- **Image Accuracy = 91.39%** --- best image-level accuracy across all variants
- **Image AUC = 0.9815** --- best image-level AUC across all variants
- Best model at epoch 41 (out of 50), showing the extra epochs were needed
- LR decayed through 5 reduction steps: 1e-3 to 3.13e-5

### Observations

- Train loss (0.11) vs val loss (0.35) gap at epoch 50 indicates overfitting is beginning
- Despite overfitting trend, the best validation metrics are still at epoch 41
- The model's per-image F1 (mean=0.66, std=0.39) remains highly variable
- High precision (0.87) but lower recall (0.70) --- the model is conservative

### Comparison with P.30 (25ep)

| Metric | P.30 (25ep) | P.30.1 (50ep) | Delta |
|--------|-------------|---------------|-------|
| Pixel F1 | 0.7438 | 0.7762 | +3.24pp |
| Pixel IoU | 0.5921 | 0.6343 | +4.22pp |
| Image Acc | 88.59% | 91.39% | +2.80pp |
| Image AUC | 0.9718 | 0.9815 | +0.97pp |

Extended training provides meaningful improvement across all metrics.


## Reproducibility Verification

This section verifies that the experiment setup is deterministic and
reproducible. It confirms seed configuration, dataset split stability,
checkpoint integrity, and records the full environment information.

In [ ]:
# ============================================================
# Reproducibility Verification
# ============================================================
import sys
import platform

print('=' * 55)
print('  REPRODUCIBILITY VERIFICATION')
print('=' * 55)

# 1. Seed Configuration
print('\n--- Seed Configuration ---')
print(f'SEED = {SEED}')
print(f'Python random:   seeded')
print(f'NumPy random:    seeded')
print(f'PyTorch manual:  seeded')
print(f'CUDA deterministic: {torch.backends.cudnn.deterministic if torch.cuda.is_available() else "N/A"}')
print(f'CUDA benchmark:    {torch.backends.cudnn.benchmark if torch.cuda.is_available() else "N/A"}')

# 2. Dataset Split Determinism
print('\n--- Dataset Split Determinism ---')
print(f'Train: {len(train_dataset)} images')
print(f'Val:   {len(val_dataset)} images')
print(f'Test:  {len(test_dataset)} images')
# Hash first 5 paths to verify determinism
import hashlib
_hash_input = '|'.join(test_dataset.image_paths[:5])
_split_hash = hashlib.md5(_hash_input.encode()).hexdigest()[:12]
print(f'Test split hash (first 5): {_split_hash}')

# 3. Checkpoint Integrity
print('\n--- Checkpoint Integrity ---')
_ckpt_dir = CHECKPOINT_DIR if isinstance(CHECKPOINT_DIR, str) else str(CHECKPOINT_DIR)
for _fname in ['best_model.pt', 'last_checkpoint.pt']:
    _path = os.path.join(_ckpt_dir, _fname)
    if os.path.exists(_path):
        _size = os.path.getsize(_path) / 1e6
        print(f'  {_fname}: {_size:.1f} MB')
    else:
        print(f'  {_fname}: not found')

# 4. Environment Information
print('\n--- Environment ---')
print(f'Python:     {sys.version.split()[0]}')
print(f'PyTorch:    {torch.__version__}')
print(f'Platform:   {platform.platform()}')
if torch.cuda.is_available():
    print(f'GPU:        {torch.cuda.get_device_name(0)}')
    print(f'VRAM:       {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
    print(f'CUDA:       {torch.version.cuda}')
try:
    import segmentation_models_pytorch as _smp
    print(f'SMP:        {_smp.__version__}')
except Exception:
    pass

print('\n' + '=' * 55)
print('  VERIFICATION COMPLETE')
print('=' * 55)

## Conclusion

vR.P.30.1 tests whether the P.30 architecture benefits from longer training (50 epochs with patience=10 vs P.30's 25 epochs with patience=7). This isolates the effect of training duration while keeping all other hyperparameters identical.

**Ablation Series Context:**
This notebook (vR.P.30.1) is part of the vR.P.30.x ablation series
systematically testing architectural and training variations:
- **P.30**: Multi-Q ELA + CBAM (baseline, 25 epochs)
- **P.30.1**: Extended training (50 epochs)
- **P.30.2**: Progressive encoder unfreezing
- **P.30.3**: Focal + Dice loss
- **P.30.4**: Geometric augmentation

**P.19** provides an orthogonal comparison using 9-channel RGB ELA input
(retaining color information) without CBAM attention.

For complete results comparison, see the Results Summary section above.

In [ ]:
# ============================================================
# 10. Save Model
# ============================================================

model_filename = f'{VERSION}_unet_resnet34_model.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_epoch': best_epoch,
    'best_val_loss': best_val_loss,
    'history': history,
    'config': {
        'version': VERSION, 'encoder': ENCODER, 'encoder_weights': ENCODER_WEIGHTS,
        'image_size': IMAGE_SIZE, 'batch_size': globals().get('BATCH_SIZE', 'N/A'),
        'learning_rate': globals().get('LEARNING_RATE', 'N/A'), 'ela_qualities': ELA_QUALITIES,
        'input_type': 'Multi-Q ELA', 'epochs_trained': len(history['train_loss']),
        'seed': SEED,
    }
}, model_filename)

print(f'Model saved: {model_filename}')
print(f'File size: {os.path.getsize(model_filename) / 1e6:.1f} MB')
# --- Save final model to Google Drive ---
import shutil as _shutil
GDRIVE_MODELS_DIR = '/content/drive/MyDrive/BigVision/models'
os.makedirs(GDRIVE_MODELS_DIR, exist_ok=True)
_gdrive_model_path = os.path.join(GDRIVE_MODELS_DIR, model_filename)
_shutil.copy2(model_filename, _gdrive_model_path)
print(f'Final model copied to Google Drive: {_gdrive_model_path}')

print(f'\n{VERSION} complete.')

# --- Save Experiment Artifacts ---
import json as _json
from datetime import datetime as _dt

# Save metrics.json
_metrics = {
    'version': VERSION,
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'epochs_trained': len(history['train_loss']),
    'history': {k: [float(v) for v in vals] for k, vals in history.items()},
}
with open(os.path.join(RESULTS_DIR, f'{VERSION}_metrics.json'), 'w') as _f:
    _json.dump(_metrics, _f, indent=2)
print(f'Metrics saved to {RESULTS_DIR}/{VERSION}_metrics.json')

# Save experiment metadata to CSV (append)
_csv_path = os.path.join(RESULTS_DIR, 'experiment_results.csv')
_row = {
    'experiment_id': VERSION,
    'run_id': f'{VERSION}_{_dt.now().strftime("%Y%m%d_%H%M%S")}',
    'dataset': 'CASIA_v2.0',
    'model': 'UNet_ResNet34',
    'seed': SEED,
    'timestamp': _dt.now().isoformat(),
    'best_epoch': best_epoch,
    'best_val_loss': float(best_val_loss),
    'epochs_trained': len(history['train_loss']),
}
try:
    _row['pixel_f1'] = float(pixel_f1)
    _row['pixel_iou'] = float(pixel_iou)
    _row['cls_accuracy'] = float(cls_accuracy)
except NameError:
    pass
_header = not os.path.exists(_csv_path)
with open(_csv_path, 'a') as _f:
    if _header:
        _f.write(','.join(_row.keys()) + '\n')
    _f.write(','.join(str(v) for v in _row.values()) + '\n')
print(f'Experiment results appended to {_csv_path}')

# Save best model to persistent location
if best_model_state is not None:
    torch.save(best_model_state, BEST_MODEL_PATH)
    print(f'Best model saved to {BEST_MODEL_PATH}')


if USE_WANDB:
    try:
        artifact = wandb.Artifact(name=f'{EXPERIMENT_ID}_{RUN_ID}_model', type='model')
        if os.path.exists(BEST_MODEL_PATH):
            artifact.add_file(BEST_MODEL_PATH)
            wandb.log_artifact(artifact)
    except Exception as e:
        print(f'W&B artifact logging failed: {e}')
    wandb.finish()
    print('W&B run finished.')


# --- HuggingFace Hub: Upload Checkpoints ---
USE_HF_HUB = True

if USE_HF_HUB:
    try:
        from huggingface_hub import HfApi, login
        from google.colab import userdata

        _hf_token = userdata.get('HF_TOKEN')
        login(token=_hf_token)
        api = HfApi()

        # Create repo if it doesn't exist
        api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=True)

        _uploaded = []
        for _fpath, _fname in [
            (BEST_MODEL_PATH, f'{VERSION}_best_model.pt'),
            (LATEST_CHECKPOINT, f'{VERSION}_latest_checkpoint.pt'),
        ]:
            if os.path.exists(_fpath):
                api.upload_file(
                    path_or_fileobj=_fpath,
                    path_in_repo=f'checkpoints/{_fname}',
                    repo_id=HF_REPO_ID,
                    commit_message=f'Upload {_fname} ({VERSION})',
                )
                _uploaded.append(_fname)

        if _uploaded:
            print(f'HuggingFace Hub: uploaded {", ".join(_uploaded)} to {HF_REPO_ID}')
        else:
            print('HuggingFace Hub: no checkpoint files found to upload')

    except Exception as e:
        print(f'HuggingFace Hub upload failed ({e}), continuing without upload')
        print('Ensure HF_TOKEN is set in Colab secrets')
